# ISC Class 12 English — Paper 1 extraction and cleaning

This notebook records the Paper 1 PDF text extraction, question parsing, answer extraction, data correction, and database-ready export workflow.

**Review note:** The source notebook contains iterative extraction/correction attempts. Cells are retained where their outputs or later variables may be dependencies. Run from top to bottom in Google Colab with the source PDF and intermediate CSVs available. For mentor review, treat the final validated/export cells as the intended output; earlier cells are exploratory or intermediate.

In [ ]:
!pip install pymupdf

from google.colab import files

uploaded = files.upload()

In [ ]:
import fitz

pdf_path = "English Paper 1 - ISC.pdf"

doc = fitz.open(pdf_path)

print("Number of pages:", len(doc))

for page_number, page in enumerate(doc, start=1):
    text = page.get_text()

    print(f"\n========== PAGE {page_number} ==========\n")
    print(text)

In [ ]:
import fitz

pdf_path = "English Paper 1 - ISC.pdf"

doc = fitz.open(pdf_path)

print("Number of pages:", len(doc))

# Create an empty variable to store all extracted text
all_text = ""

for page_number, page in enumerate(doc, start=1):
    text = page.get_text()

    print(f"\n========== PAGE {page_number} ==========\n")
    print(text)

    # Add page number and extracted text to all_text
    all_text += f"\n========== PAGE {page_number} ==========\n"
    all_text += text

# Save the extracted text into a text file
with open("English_Paper_1_raw.txt", "w", encoding="utf-8") as file:
    file.write(all_text)

print("\nExtraction completed and saved!")

In [ ]:
pages_data = []

for page_number, page in enumerate(doc, start=1):
    text = page.get_text()

    pages_data.append({
        "page_number": page_number,
        "text": text
    })

print(pages_data[0])

In [ ]:
import re

clean_text = all_text

# Remove repeated page headers
clean_text = re.sub(
    r'ISC.*?Class-XII',
    '',
    clean_text
)

# Remove "SOLVED PAPER - 2025"
clean_text = re.sub(
    r'SOLVED PAPER\s*-\s*2025',
    '',
    clean_text
)

# Remove excessive blank lines
clean_text = re.sub(
    r'\n\s*\n+',
    '\n\n',
    clean_text
)

# Remove unnecessary spaces at the beginning/end of lines
clean_text = '\n'.join(
    line.strip() for line in clean_text.splitlines()
)

print(clean_text[:5000])

In [ ]:
import re

clean_text = all_text

# Remove repeated spaces/tabs
clean_text = re.sub(r'[ \t]+', ' ', clean_text)

# Remove excessive blank lines
clean_text = re.sub(r'\n\s*\n+', '\n\n', clean_text)

# Remove leading/trailing spaces from each line
clean_text = '\n'.join(line.strip() for line in clean_text.splitlines())

# Save cleaned text
with open("English_Paper_1_cleaned.txt", "w", encoding="utf-8") as file:
    file.write(clean_text)

print("Cleaning completed!")

In [ ]:
import re

# Find Question 1, Question 2, Question 3, etc.
question_pattern = r'Question\s+(\d+)'

matches = list(re.finditer(question_pattern, clean_text))

print("Number of main questions found:", len(matches))

for match in matches:
    print("Question:", match.group(1), "Position:", match.start())

In [ ]:
questions = {}

for i, match in enumerate(matches):
    question_number = match.group(1)

    start = match.start()

    if i + 1 < len(matches):
        end = matches[i + 1].start()
    else:
        end = len(clean_text)

    question_text = clean_text[start:end].strip()

    questions[f"Question {question_number}"] = question_text


for question_number, text in questions.items():
    print("\n" + "=" * 60)
    print(question_number)
    print("=" * 60)
    print(text[:2000])

In [ ]:
import re

q1_text = questions["Question 1"]

# Find (i), (ii), (iii), etc.
sub_pattern = r'\(([ivx]+)\)'

matches = list(re.finditer(sub_pattern, q1_text))

q1_subquestions = {}

for i, match in enumerate(matches):
    sub_number = match.group(1)

    start = match.end()

    if i + 1 < len(matches):
        end = matches[i + 1].start()
    else:
        end = len(q1_text)

    text = q1_text[start:end].strip()

    q1_subquestions[sub_number] = text


for number, text in q1_subquestions.items():
    print("\n" + "=" * 50)
    print(f"Question 1({number})")
    print("=" * 50)
    print(text)

In [ ]:
q2_text = questions["Question 2"]

sub_pattern = r'\(([ivx]+)\)'

matches = list(re.finditer(sub_pattern, q2_text))

q2_subquestions = {}

for i, match in enumerate(matches):
    sub_number = match.group(1)

    start = match.end()

    if i + 1 < len(matches):
        end = matches[i + 1].start()
    else:
        end = len(q2_text)

    text = q2_text[start:end].strip()

    q2_subquestions[sub_number] = text


for number, text in q2_subquestions.items():
    print("\n" + "=" * 50)
    print(f"Question 2({number})")
    print("=" * 50)
    print(text)

In [ ]:
import re

# Get Question 3
q3_text = questions["Question 3"]

# Remove page headers and footers
q3_text = re.sub(r'========== PAGE \d+ ==========', '', q3_text)
q3_text = re.sub(r'SOLVED PAPER\s*-\s*2025\s*\(\s*\d+\s*\)', '', q3_text)
q3_text = re.sub(r'ISC,\s*ENGLISH PAPER-1.*?\(\s*\d+\s*\)', '', q3_text)

# Find Q3(i)
start = q3_text.find("(i) Rewrite the following sentences")
end = q3_text.find("(ii) Replace the INCORRECT")

if start == -1 or end == -1:
    print("Could not find Q3(i).")

else:
    q3_i_text = q3_text[start:end].strip()

    # Find (a) to (e)
    matches = list(re.finditer(r'\(([a-e])\)', q3_i_text))

    # Store questions
    q3_i_data = []

    for i, match in enumerate(matches):

        letter = match.group(1)

        start_pos = match.end()

        if i + 1 < len(matches):
            end_pos = matches[i + 1].start()
        else:
            end_pos = len(q3_i_text)

        question_text = q3_i_text[start_pos:end_pos].strip()

        q3_i_data.append({
            "question_number": f"3(i)({letter})",
            "question_text": question_text
        })

    # Display the extracted data
    print("=" * 70)
    print("QUESTION 3(i)")
    print("=" * 70)

    for item in q3_i_data:
        print("\nQuestion Number:", item["question_number"])
        print("Question Text:", item["question_text"])

In [ ]:
import re

# Get Question 3 directly
q3_text = questions["Question 3"]

# Remove page headers and footers
q3_text = re.sub(r'========== PAGE \d+ ==========', '', q3_text)
q3_text = re.sub(r'SOLVED PAPER\s*-\s*2025\s*\(\s*\d+\s*\)', '', q3_text)
q3_text = re.sub(r'ISC,\s*ENGLISH PAPER-1.*?\(\s*\d+\s*\)', '', q3_text)

# Find Q3(ii)
start = q3_text.find("(ii) Replace the INCORRECT")
end = q3_text.find("(iii) Fill in the blanks")

if start == -1 or end == -1:
    print("Could not find Q3(ii).")

else:
    q3_ii_text = q3_text[start:end].strip()

    # Find (a) to (j)
    matches = list(re.finditer(r'\(([a-j])\)', q3_ii_text))

    print("=" * 70)
    print("QUESTION 3(ii)")
    print("=" * 70)

    for i, match in enumerate(matches):

        letter = match.group(1)

        start_pos = match.end()

        if i + 1 < len(matches):
            end_pos = matches[i + 1].start()
        else:
            end_pos = len(q3_ii_text)

        question = q3_ii_text[start_pos:end_pos].strip()

        print(f"\nQ3(ii)({letter})")
        print("-" * 50)
        print(question)

In [ ]:
import re

# Get Question 3 directly
q3_text = questions["Question 3"]

# Remove page headers and footers
q3_text = re.sub(r'========== PAGE \d+ ==========', '', q3_text)
q3_text = re.sub(r'SOLVED PAPER\s*-\s*2025\s*\(\s*\d+\s*\)', '', q3_text)
q3_text = re.sub(r'ISC,\s*ENGLISH PAPER-1.*?\(\s*\d+\s*\)', '', q3_text)

# Find Q3(iii)
start = q3_text.find("(iii) Fill in the blanks")

if start == -1:
    print("Could not find Q3(iii).")

else:
    q3_iii_text = q3_text[start:].strip()

    # Find blanks (1) to (10)
    matches = list(re.finditer(r'\((\d+)\)\s*\(([^)]+)\)', q3_iii_text))

    print("=" * 70)
    print("QUESTION 3(iii)")
    print("=" * 70)

    for match in matches:

        number = match.group(1)
        verb = match.group(2)

        print(f"\nBlank {number}")
        print("-" * 50)
        print("Verb given:", verb)

In [ ]:
import re
import pandas as pd

# Get Question 3
q3_text = questions["Question 3"]

# Remove page headers and footers
q3_text = re.sub(r'========== PAGE \d+ ==========', '', q3_text)
q3_text = re.sub(r'SOLVED PAPER\s*-\s*2025\s*\(\s*\d+\s*\)', '', q3_text)

# Store all Q3 questions
q3_data = []

# ============================================================
# Q3(i) - (a) to (e)
# ============================================================

start = q3_text.find("(i) Rewrite the following sentences")
end = q3_text.find("(ii) Replace the INCORRECT")

q3_i_text = q3_text[start:end]

matches = list(re.finditer(r'\(([a-e])\)', q3_i_text))

for i, match in enumerate(matches):

    letter = match.group(1)
    start_pos = match.end()

    if i + 1 < len(matches):
        end_pos = matches[i + 1].start()
    else:
        end_pos = len(q3_i_text)

    question_text = q3_i_text[start_pos:end_pos].strip()

    q3_data.append({
        "question_number": f"3(i)({letter})",
        "question_text": question_text
    })


# ============================================================
# Q3(ii) - (a) to (j)
# ============================================================

start = q3_text.find("(ii) Replace the INCORRECT")
end = q3_text.find("(iii) Fill in the blanks")

q3_ii_text = q3_text[start:end]

matches = list(re.finditer(r'\(([a-j])\)', q3_ii_text))

for i, match in enumerate(matches):

    letter = match.group(1)
    start_pos = match.end()

    if i + 1 < len(matches):
        end_pos = matches[i + 1].start()
    else:
        end_pos = len(q3_ii_text)

    question_text = q3_ii_text[start_pos:end_pos].strip()

    q3_data.append({
        "question_number": f"3(ii)({letter})",
        "question_text": question_text
    })


# ============================================================
# Q3(iii) - (1) to (10)
# ============================================================

start = q3_text.find("(iii) Fill in the blanks")

q3_iii_text = q3_text[start:]

matches = list(
    re.finditer(r'\((\d+)\)\s*\(([^)]+)\)', q3_iii_text)
)

for match in matches:

    number = match.group(1)
    verb = match.group(2)

    q3_data.append({
        "question_number": f"3(iii)({number})",
        "question_text": f"Given verb: {verb}"
    })


# ============================================================
# CREATE DATAFRAME
# ============================================================

q3_df = pd.DataFrame(q3_data)

print("=" * 70)
print("QUESTION 3 DATA")
print("=" * 70)

print(q3_df.to_string(index=False))


# ============================================================
# SAVE CSV
# ============================================================

q3_df.to_csv("ISC_English_2025_Q3.csv", index=False)

print("\nQ3 data saved successfully!")

In [ ]:
import re

# Get Question 4
q4_text = questions["Question 4"]

# Remove page headers and footers
q4_text = re.sub(r'========== PAGE \d+ ==========', '', q4_text)
q4_text = re.sub(r'SOLVED PAPER\s*-\s*2025\s*\(\s*\d+\s*\)', '', q4_text)
q4_text = re.sub(r'ISC,\s*ENGLISH PAPER-1.*?\(\s*\d+\s*\)', '', q4_text)

# Find Q4(i)
start = q4_text.find("(i) (a) Find a single word")
end = q4_text.find("(ii) Answer the following questions")

if start == -1 or end == -1:
    print("Could not find Q4(i).")

else:
    q4_i_text = q4_text[start:end].strip()

    print("=" * 70)
    print("QUESTION 4(i)")
    print("=" * 70)
    print(q4_i_text)

In [ ]:
import re

# Q4(i) is already extracted
q4_i_text = q4_i_text

# ------------------------------------------------------------
# Find section (a) and section (b)
# ------------------------------------------------------------

start_a = q4_i_text.find("(a) Find a single word")
start_b = q4_i_text.find("(b)")

q4_i_a = q4_i_text[start_a:start_b].strip()
q4_i_b = q4_i_text[start_b:].strip()

# ------------------------------------------------------------
# Extract Q4(i)(a) - questions (1), (2), (3)
# ------------------------------------------------------------

a_matches = list(re.finditer(r'\((1|2|3)\)', q4_i_a))

print("=" * 70)
print("QUESTION 4(i)(a)")
print("=" * 70)

for i, match in enumerate(a_matches):

    number = match.group(1)

    start_pos = match.end()

    if i + 1 < len(a_matches):
        end_pos = a_matches[i + 1].start()
    else:
        end_pos = len(q4_i_a)

    question = q4_i_a[start_pos:end_pos].strip()

    print(f"\nQ4(i)(a)({number})")
    print("-" * 50)
    print(question)


# ------------------------------------------------------------
# Extract Q4(i)(b) - questions (1), (2), (3)
# ------------------------------------------------------------

b_matches = list(re.finditer(r'\((1|2|3)\)', q4_i_b))

print("\n" + "=" * 70)
print("QUESTION 4(i)(b)")
print("=" * 70)

for i, match in enumerate(b_matches):

    number = match.group(1)

    start_pos = match.end()

    if i + 1 < len(b_matches):
        end_pos = b_matches[i + 1].start()
    else:
        end_pos = len(q4_i_b)

    question = q4_i_b[start_pos:end_pos].strip()

    print(f"\nQ4(i)(b)({number})")
    print("-" * 50)
    print(question)


In [ ]:
import re

# Get Question 4
q4_text = questions["Question 4"]

# Remove page headers and footers
q4_text = re.sub(r'========== PAGE \d+ ==========', '', q4_text)
q4_text = re.sub(r'SOLVED PAPER\s*-\s*2025\s*\(\s*\d+\s*\)', '', q4_text)

# Find Q4(ii)
start = q4_text.find("(ii) Answer the following questions")
end = q4_text.find("(iii) Summarise Emma")

if start == -1 or end == -1:
    print("Could not find Q4(ii).")

else:
    q4_ii_text = q4_text[start:end].strip()

    # Find (a), (b), (c)
    matches = list(re.finditer(r'\(([a-c])\)', q4_ii_text))

    print("=" * 70)
    print("QUESTION 4(ii)")
    print("=" * 70)

    for i, match in enumerate(matches):

        letter = match.group(1)

        start_pos = match.end()

        if i + 1 < len(matches):
            end_pos = matches[i + 1].start()
        else:
            end_pos = len(q4_ii_text)

        question = q4_ii_text[start_pos:end_pos].strip()

        print(f"\nQ4(ii)({letter})")
        print("-" * 50)
        print(question)

In [ ]:
import re

# Get Question 4
q4_text = questions["Question 4"]

# Remove page headers and footers
q4_text = re.sub(r'========== PAGE \d+ ==========', '', q4_text)
q4_text = re.sub(r'SOLVED PAPER\s*-\s*2025\s*\(\s*\d+\s*\)', '', q4_text)

# Find Q4(iii)
start = q4_text.find("(iii) Summarise Emma")

# Stop before the Answers section
end = q4_text.find("Answers")

if start == -1:
    print("Could not find Q4(iii).")
else:
    if end == -1:
        end = len(q4_text)

    q4_iii_text = q4_text[start:end].strip()

    print("=" * 70)
    print("QUESTION 4(iii)")
    print("=" * 70)
    print(q4_iii_text)

In [ ]:
# Find the Answers section
answers_start = questions["Question 4"].find("Answers")

if answers_start == -1:
    print("Answers section not found.")
else:
    answers_text = questions["Question 4"][answers_start:].strip()

    print("=" * 70)
    print("ANSWERS SECTION")
    print("=" * 70)

    print(answers_text[:8000])

In [ ]:
# Find the Answers section
answers_text = questions["Question 4"][
    questions["Question 4"].find("Answers"):
]

# Find Q1 answer
start = answers_text.find("1. (i)")
end = answers_text.find("2. (i)")

if start == -1 or end == -1:
    print("Could not find Q1 answers.")
else:
    q1_answers_text = answers_text[start:end].strip()

    print("=" * 70)
    print("QUESTION 1 - ANSWERS")
    print("=" * 70)
    print(q1_answers_text)

In [ ]:
import re

# Get the Answers section from Question 4
full_text = questions["Question 4"]

answers_start = full_text.find("Answers")

if answers_start == -1:
    print("Answers section not found.")
else:
    answers_text = full_text[answers_start:]

    # Extract only Question 1 answers
    q1_start = answers_text.find("1. (i)")
    q2_start = answers_text.find("2. (i)")

    if q1_start == -1:
        print("Question 1 answers not found.")
    else:
        if q2_start != -1:
            q1_text = answers_text[q1_start:q2_start]
        else:
            q1_text = answers_text[q1_start:]

        # Remove PDF page headers/footers
        q1_text = re.sub(
            r'={5,} PAGE \d+ ={5,}',
            '',
            q1_text
        )

        q1_text = re.sub(
            r'ISC,?\s*ENGLISH PAPER-1 \(Language\), Class-XII',
            '',
            q1_text,
            flags=re.IGNORECASE
        )

        q1_text = re.sub(
            r'SOLVED PAPER\s*-\s*2025\s*\(\s*\d+\s*\)',
            '',
            q1_text,
            flags=re.IGNORECASE
        )

        # Find positions of (i), (ii), (iii), (iv), (v), (vi)
        pattern = r'\n?\((i|ii|iii|iv|v|vi)\)\s+'
        matches = list(re.finditer(pattern, q1_text))

        q1_answers = []

        for index, match in enumerate(matches):
            sub_question = f"({match.group(1)})"

            start = match.end()

            if index + 1 < len(matches):
                end = matches[index + 1].start()
            else:
                end = len(q1_text)

            answer = q1_text[start:end].strip()

            # Clean extra spaces/newlines
            answer = re.sub(r'\s+', ' ', answer).strip()

            q1_answers.append({
                "question_id": f"ENG25-P1-Q1-{match.group(1)}",
                "parent_question": "Q1",
                "sub_question": sub_question,
                "question_type": "Composition",
                "marks": 20,
                "answer": answer
            })

        # Display results
        print("=" * 80)
        print("QUESTION 1 - SEGMENTED ANSWERS")
        print("=" * 80)

        for item in q1_answers:
            print(f"\n{item['sub_question']}")
            print("-" * 80)
            print(item["answer"])

        print("\nTotal answers extracted:", len(q1_answers))

In [ ]:
# Clean page-number artifacts from Q1 answers

for item in q1_answers:
    answer = item["answer"]

    # Remove page-number artifacts such as (16), (18)
    answer = re.sub(r'\(\s*\d+\s*\)', '', answer)

    # Clean extra spaces created after removal
    answer = re.sub(r'\s+', ' ', answer).strip()

    item["answer"] = answer


# Display cleaned answers
print("=" * 80)
print("CLEANED QUESTION 1 ANSWERS")
print("=" * 80)

for item in q1_answers:
    print(f"\n{item['question_id']}")
    print("-" * 80)
    print(item["answer"])

In [ ]:
import re

# Get the full Answers section
full_text = questions["Question 4"]

answers_start = full_text.find("Answers")

if answers_start == -1:
    print("Answers section not found.")
else:
    answers_text = full_text[answers_start:]

    # Find the beginning of Q2 and Q3 answers
    q2_start = answers_text.find("2. (i)")
    q3_start = answers_text.find("3. (i)")

    if q2_start == -1:
        print("Question 2 answers not found.")
    else:
        if q3_start != -1:
            q2_text = answers_text[q2_start:q3_start]
        else:
            q2_text = answers_text[q2_start:]

        # Remove PDF page headers/footers
        q2_text = re.sub(
            r'={5,} PAGE \d+ ={5,}',
            '',
            q2_text
        )

        q2_text = re.sub(
            r'ISC,?\s*ENGLISH PAPER-1 \(Language\), Class-XII',
            '',
            q2_text,
            flags=re.IGNORECASE
        )

        q2_text = re.sub(
            r'SOLVED PAPER\s*-\s*2025\s*\(\s*\d+\s*\)',
            '',
            q2_text,
            flags=re.IGNORECASE
        )

        # Remove PDF page-number artifacts such as (19), (20), etc.
        q2_text = re.sub(r'\(\s*\d+\s*\)', '', q2_text)

        # Find Q2(i) and Q2(ii)
        pattern = r'\n?\((i|ii)\)\s+'
        matches = list(re.finditer(pattern, q2_text))

        q2_answers = []

        for index, match in enumerate(matches):
            sub_question = f"({match.group(1)})"

            start = match.end()

            if index + 1 < len(matches):
                end = matches[index + 1].start()
            else:
                end = len(q2_text)

            answer = q2_text[start:end].strip()

            # Clean extra whitespace
            answer = re.sub(r'\s+', ' ', answer).strip()

            q2_answers.append({
                "question_id": f"ENG25-P1-Q2-{match.group(1)}",
                "parent_question": "Q2",
                "sub_question": sub_question,
                "question_type": "Writing",
                "answer": answer
            })

        # Display results
        print("=" * 80)
        print("QUESTION 2 - SEGMENTED ANSWERS")
        print("=" * 80)

        for item in q2_answers:
            print(f"\n{item['question_id']}")
            print("-" * 80)
            print(item["answer"])

        print("\nTotal answers extracted:", len(q2_answers))

In [ ]:
# Clean Q2 answers

for item in q2_answers:
    answer = item["answer"]

    # Remove PDF page-number artifacts
    answer = re.sub(r'\(\s*\d+\s*\)', '', answer)

    # Clean extra whitespace
    answer = re.sub(r'\s+', ' ', answer).strip()

    item["answer"] = answer


# Display cleaned Q2 answers
print("=" * 80)
print("CLEANED QUESTION 2 ANSWERS")
print("=" * 80)

for item in q2_answers:
    print(f"\n{item['question_id']}")
    print("-" * 80)
    print(item["answer"])

print("\nTotal answers:", len(q2_answers))

In [ ]:
import re

# Get the full Answers section
full_text = questions["Question 4"]

answers_start = full_text.find("Answers")

if answers_start == -1:
    print("Answers section not found.")
else:
    answers_text = full_text[answers_start:]

    # Find Q3 and Q4 answer boundaries
    q3_start = answers_text.find("3. (i)")
    q4_start = answers_text.find("4. (i)")

    if q3_start == -1:
        print("Question 3 answers not found.")
    else:
        if q4_start != -1:
            q3_text = answers_text[q3_start:q4_start]
        else:
            q3_text = answers_text[q3_start:]

        # Remove PDF page headers/footers
        q3_text = re.sub(
            r'={5,} PAGE \d+ ={5,}',
            '',
            q3_text
        )

        q3_text = re.sub(
            r'ISC,?\s*ENGLISH PAPER-1 \(Language\), Class-XII',
            '',
            q3_text,
            flags=re.IGNORECASE
        )

        q3_text = re.sub(
            r'SOLVED PAPER\s*-\s*2025\s*\(\s*\d+\s*\)',
            '',
            q3_text,
            flags=re.IGNORECASE
        )

        # Remove PDF page-number artifacts
        q3_text = re.sub(r'\(\s*\d+\s*\)', '', q3_text)

        # -------------------------------------------------
        # Extract Q3(i) - Sentence Transformations
        # -------------------------------------------------

        q3_i_start = q3_text.find("(i)")
        q3_ii_start = q3_text.find("(ii)")

        q3_answers = []

        if q3_i_start != -1 and q3_ii_start != -1:

            q3_i_text = q3_text[q3_i_start:q3_ii_start]

            # Find (a), (b), (c), (d), (e)
            matches = list(
                re.finditer(r'\(([a-e])\)\s+', q3_i_text)
            )

            for index, match in enumerate(matches):

                sub_question = f"({match.group(1)})"

                start = match.end()

                if index + 1 < len(matches):
                    end = matches[index + 1].start()
                else:
                    end = len(q3_i_text)

                answer = q3_i_text[start:end].strip()
                answer = re.sub(r'\s+', ' ', answer).strip()

                q3_answers.append({
                    "question_id": f"ENG25-P1-Q3-i-{match.group(1)}",
                    "parent_question": "Q3",
                    "section": "(i)",
                    "sub_question": sub_question,
                    "question_type": "Sentence Transformation",
                    "answer": answer
                })

        # -------------------------------------------------
        # Extract Q3(ii) - Phrasal Verbs
        # -------------------------------------------------

        q3_ii_start = q3_text.find("(ii)")
        q3_iii_start = q3_text.find("(iii)")

        if q3_ii_start != -1 and q3_iii_start != -1:

            q3_ii_text = q3_text[q3_ii_start:q3_iii_start]

            # Find (a) to (j)
            matches = list(
                re.finditer(r'\(([a-j])\)\s+', q3_ii_text)
            )

            for index, match in enumerate(matches):

                sub_question = f"({match.group(1)})"

                start = match.end()

                if index + 1 < len(matches):
                    end = matches[index + 1].start()
                else:
                    end = len(q3_ii_text)

                answer = q3_ii_text[start:end].strip()
                answer = re.sub(r'\s+', ' ', answer).strip()

                q3_answers.append({
                    "question_id": f"ENG25-P1-Q3-ii-{match.group(1)}",
                    "parent_question": "Q3",
                    "section": "(ii)",
                    "sub_question": sub_question,
                    "question_type": "Phrasal Verb",
                    "answer": answer
                })

        # -------------------------------------------------
        # Extract Q3(iii) - Verb Forms
        # -------------------------------------------------

        q3_iii_start = q3_text.find("(iii)")

        if q3_iii_start != -1:

            q3_iii_text = q3_text[q3_iii_start:]

            # Find numbered answers 1 to 10
            matches = list(
                re.finditer(r'(?:^|\s)([1-9]|10)\.\s+', q3_iii_text)
            )

            for index, match in enumerate(matches):

                question_number = match.group(1)

                start = match.end()

                if index + 1 < len(matches):
                    end = matches[index + 1].start()
                else:
                    end = len(q3_iii_text)

                answer = q3_iii_text[start:end].strip()
                answer = re.sub(r'\s+', ' ', answer).strip()

                q3_answers.append({
                    "question_id": f"ENG25-P1-Q3-iii-{question_number}",
                    "parent_question": "Q3",
                    "section": "(iii)",
                    "sub_question": question_number,
                    "question_type": "Verb Form",
                    "answer": answer
                })

        # -------------------------------------------------
        # Display results
        # -------------------------------------------------

        print("=" * 80)
        print("QUESTION 3 - SEGMENTED ANSWERS")
        print("=" * 80)

        for item in q3_answers:
            print(f"\n{item['question_id']}")
            print("-" * 80)
            print(item["answer"])

        print("\nTotal answers extracted:", len(q3_answers))

In [ ]:
# Clean Q3 answers

for item in q3_answers:
    answer = item["answer"]

    # Remove PDF page-number artifacts such as (19), (20), etc.
    answer = re.sub(r'\(\s*\d+\s*\)', '', answer)

    # Clean extra whitespace
    answer = re.sub(r'\s+', ' ', answer).strip()

    item["answer"] = answer


# Display cleaned Q3 answers
print("=" * 80)
print("CLEANED QUESTION 3 ANSWERS")
print("=" * 80)

for item in q3_answers:
    print(f"\n{item['question_id']}")
    print("-" * 80)
    print(item["answer"])

print("\nTotal answers:", len(q3_answers))

In [ ]:
import re

# q4_text is already available from the previous step

q4_answers = []

# ============================================================
# Q4(i)(a) - Single Word Replacement
# ============================================================

ia_start = q4_text.find("(i) (a)")
ib_start = q4_text.find("(i) (b)")

if ia_start != -1 and ib_start != -1:

    ia_text = q4_text[ia_start:ib_start]

    matches = list(
        re.finditer(r'(?m)^\s*([1-3])\.\s*(.+)$', ia_text)
    )

    for match in matches:

        number = match.group(1)
        answer = match.group(2).strip()

        answer = re.sub(r'\s+', ' ', answer)

        q4_answers.append({
            "question_id": f"ENG25-P1-Q4-i-a-{number}",
            "parent_question": "Q4",
            "section": "(i)(a)",
            "sub_question": number,
            "question_type": "Single Word Replacement",
            "answer": answer
        })


# ============================================================
# Q4(i)(b) - MCQ
# ============================================================

ib_start = q4_text.find("(i) (b)")
ii_start = q4_text.find("(ii)")

if ib_start != -1 and ii_start != -1:

    ib_text = q4_text[ib_start:ii_start]

    # Split using 1., 2., 3.
    matches = list(
        re.finditer(r'(?m)^\s*([1-3])\.\s*', ib_text)
    )

    for index, match in enumerate(matches):

        number = match.group(1)

        start = match.end()

        if index + 1 < len(matches):
            end = matches[index + 1].start()
        else:
            end = len(ib_text)

        answer = ib_text[start:end].strip()

        # Remove unnecessary line breaks
        answer = re.sub(r'\s+', ' ', answer)

        q4_answers.append({
            "question_id": f"ENG25-P1-Q4-i-b-{number}",
            "parent_question": "Q4",
            "section": "(i)(b)",
            "sub_question": number,
            "question_type": "MCQ",
            "answer": answer
        })


# ============================================================
# Q4(ii) - Short Answers
# ============================================================

ii_start = q4_text.find("(ii)")
iii_start = q4_text.find("(iii)")

if ii_start != -1 and iii_start != -1:

    ii_text = q4_text[ii_start:iii_start]

    matches = list(
        re.finditer(r'(?m)^\s*\(([a-c])\)\s*', ii_text)
    )

    for index, match in enumerate(matches):

        letter = match.group(1)

        start = match.end()

        if index + 1 < len(matches):
            end = matches[index + 1].start()
        else:
            end = len(ii_text)

        answer = ii_text[start:end].strip()

        answer = re.sub(r'\s+', ' ', answer)

        q4_answers.append({
            "question_id": f"ENG25-P1-Q4-ii-{letter}",
            "parent_question": "Q4",
            "section": "(ii)",
            "sub_question": f"({letter})",
            "question_type": "Short Answer",
            "answer": answer
        })


# ============================================================
# Q4(iii) - Summary
# ============================================================

if iii_start != -1:

    iii_text = q4_text[iii_start:]

    # Remove the (iii) heading
    iii_text = re.sub(
        r'^\(iii\)\s*',
        '',
        iii_text
    )

    answer = re.sub(r'\s+', ' ', iii_text).strip()

    q4_answers.append({
        "question_id": "ENG25-P1-Q4-iii",
        "parent_question": "Q4",
        "section": "(iii)",
        "sub_question": "(iii)",
        "question_type": "Summary",
        "answer": answer
    })


# ============================================================
# DISPLAY RESULTS
# ============================================================

print("=" * 80)
print("QUESTION 4 - SEGMENTED ANSWERS")
print("=" * 80)

for item in q4_answers:

    print(f"\n{item['question_id']}")
    print("-" * 80)
    print(item["answer"])

print("\nTotal answers extracted:", len(q4_answers))

In [ ]:
# Re-extract Q4(ii) correctly

ii_start = q4_text.find("(ii)")
iii_start = q4_text.find("(iii)")

if ii_start == -1 or iii_start == -1:
    print("Q4(ii) section not found.")
else:
    q4_ii_text = q4_text[ii_start:iii_start]

    # Remove the (ii) heading
    q4_ii_text = re.sub(r'^\(ii\)\s*', '', q4_ii_text)

    q4_ii_answers = []

    # Find (a), (b), and (c)
    matches = list(
        re.finditer(r'\(([a-c])\)\s*', q4_ii_text)
    )

    for index, match in enumerate(matches):

        letter = match.group(1)

        start = match.end()

        if index + 1 < len(matches):
            end = matches[index + 1].start()
        else:
            end = len(q4_ii_text)

        answer = q4_ii_text[start:end].strip()

        # Clean whitespace
        answer = re.sub(r'\s+', ' ', answer)

        q4_ii_answers.append({
            "question_id": f"ENG25-P1-Q4-ii-{letter}",
            "parent_question": "Q4",
            "section": "(ii)",
            "sub_question": f"({letter})",
            "question_type": "Short Answer",
            "answer": answer
        })

    # Display
    print("=" * 80)
    print("Q4(ii) - SEGMENTED ANSWERS")
    print("=" * 80)

    for item in q4_ii_answers:
        print(f"\n{item['question_id']}")
        print("-" * 80)
        print(item["answer"])

    print("\nTotal Q4(ii) answers:", len(q4_ii_answers))

In [ ]:
import fitz
import re
import pandas as pd

# ============================================================
# 1. OPEN PDF
# ============================================================

pdf_path = "English Paper 1 - ISC.pdf"

doc = fitz.open(pdf_path)

all_text = ""

for page in doc:
    all_text += page.get_text() + "\n"


# ============================================================
# 2. CLEAN BASIC PDF TEXT
# ============================================================

text = all_text

# Remove repeated page numbers such as (16), (18), etc.
text = re.sub(r'\(\s*\d+\s*\)', '', text)

# Normalize spaces
text = re.sub(r'[ \t]+', ' ', text)

# Normalize multiple newlines
text = re.sub(r'\n+', '\n', text)


# ============================================================
# 3. FIND QUESTION 1
# ============================================================

# Q1 starts with "1. (i)"
q1_match = re.search(
    r'1\.\s*\(i\)',
    text,
    re.IGNORECASE
)

# Q2 starts with "2. (i)"
q2_match = re.search(
    r'2\.\s*\(i\)',
    text,
    re.IGNORECASE
)

if not q1_match:
    print("❌ Q1 could not be found.")
else:

    q1_start = q1_match.start()

    if q2_match:
        q1_end = q2_match.start()
    else:
        q1_end = len(text)

    q1_text = text[q1_start:q1_end]


    # ========================================================
    # 4. EXTRACT Q1(i) TO Q1(vi)
    # ========================================================

    pattern = r'\((i|ii|iii|iv|v|vi)\)\s*'

    matches = list(
        re.finditer(pattern, q1_text, re.IGNORECASE)
    )

    q1_questions = []

    for i, match in enumerate(matches):

        sub_question = f"({match.group(1)})"

        start = match.end()

        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(q1_text)

        question = q1_text[start:end]

        # Clean question text
        question = re.sub(r'\s+', ' ', question).strip()

        q1_questions.append({
            "sub_question": sub_question,
            "question_text": question
        })


    # ========================================================
    # 5. FIND ANSWERS SECTION
    # ========================================================

    answers_match = re.search(
        r'Answers',
        text,
        re.IGNORECASE
    )

    q1_answers = []

    if answers_match:

        answers_text = text[answers_match.start():]

        # Find Q1 answer section
        answer_q1_match = re.search(
            r'1\.\s*\(i\)',
            answers_text,
            re.IGNORECASE
        )

        answer_q2_match = re.search(
            r'2\.\s*\(i\)',
            answers_text,
            re.IGNORECASE
        )

        if answer_q1_match:

            answer_start = answer_q1_match.start()

            if answer_q2_match:
                answer_end = answer_q2_match.start()
            else:
                answer_end = len(answers_text)

            q1_answer_text = answers_text[
                answer_start:answer_end
            ]


            # =================================================
            # 6. EXTRACT SIX ANSWERS
            # =================================================

            answer_matches = list(
                re.finditer(
                    pattern,
                    q1_answer_text,
                    re.IGNORECASE
                )
            )

            for i, match in enumerate(answer_matches):

                sub_question = f"({match.group(1)})"

                start = match.end()

                if i + 1 < len(answer_matches):
                    end = answer_matches[i + 1].start()
                else:
                    end = len(q1_answer_text)

                answer = q1_answer_text[start:end]

                # Clean answer
                answer = re.sub(
                    r'\s+',
                    ' ',
                    answer
                ).strip()

                q1_answers.append({
                    "sub_question": sub_question,
                    "answer": answer
                })


    # ========================================================
    # 7. DISPLAY EXTRACTION COUNTS
    # ========================================================

    print("=" * 80)
    print("EXTRACTION CHECK")
    print("=" * 80)

    print("Q1 questions found:", len(q1_questions))
    print("Q1 answers found:", len(q1_answers))


    # ========================================================
    # 8. MAP QUESTION + ANSWER
    # ========================================================

    mapped_q1 = []

    for question in q1_questions:

        sub_question = question["sub_question"]

        matching_answer = ""

        for answer_item in q1_answers:

            if answer_item["sub_question"].lower() == sub_question.lower():

                matching_answer = answer_item["answer"]

                break


        mapped_q1.append({

            "question_id":
                f"ENG25-P1-Q1-{sub_question.strip('()')}",

            "board": "ISC",

            "class": "Class XII",

            "subject": "English",

            "year": 2025,

            "paper":
                "Paper 1 - English Language",

            "source_type":
                "Solved Paper",

            "parent_question":
                "Q1",

            "sub_question":
                sub_question,

            "question_type":
                "Composition",

            "marks":
                20,

            "question_text":
                question["question_text"],

            "answer":
                matching_answer,

            "source_file":
                pdf_path
        })


    # ========================================================
    # 9. CREATE DATAFRAME
    # ========================================================

    q1_df = pd.DataFrame(mapped_q1)


    # ========================================================
    # 10. DISPLAY RESULT
    # ========================================================

    print("\n" + "=" * 80)
    print("Q1 QUESTION-ANSWER MAPPING")
    print("=" * 80)

    print("\nTotal Q1 records:", len(q1_df))

    if len(q1_df) > 0:
        display(q1_df)
    else:
        print("❌ No Q1 records were created.")


    # ========================================================
    # 11. VALIDATION
    # ========================================================

    print("\n" + "=" * 80)
    print("VALIDATION")
    print("=" * 80)

    if "answer" in q1_df.columns:

        missing_answers = q1_df[
            q1_df["answer"].fillna("").str.strip() == ""
        ]

        if len(missing_answers) == 0:

            print(
                "✅ All Q1 questions have corresponding answers."
            )

        else:

            print(
                "⚠️ Missing answers:",
                len(missing_answers)
            )

            display(missing_answers)

    else:

        print("❌ Answer column was not created.")


    # ========================================================
    # 12. SAVE CSV
    # ========================================================

    output_file = "ISC_English_Paper1_2025_Q1.csv"

    q1_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    print("\nCSV saved as:", output_file)

In [ ]:
import fitz
import re
import pandas as pd

# ============================================================
# 1. OPEN PDF
# ============================================================

pdf_path = "English Paper 1 - ISC.pdf"

doc = fitz.open(pdf_path)

all_text = ""

for page in doc:
    all_text += page.get_text() + "\n"


# ============================================================
# 2. BASIC CLEANING
# ============================================================

text = all_text

# Remove PDF page-number artifacts such as (16), (18), etc.
text = re.sub(r'\(\s*\d+\s*\)', '', text)

# Normalize spaces
text = re.sub(r'[ \t]+', ' ', text)

# Normalize new lines
text = re.sub(r'\n+', '\n', text)


# ============================================================
# 3. FIND QUESTION 2
# ============================================================

# Q2 starts with 2. (i)
q2_start_match = re.search(
    r'2\.\s*\(i\)',
    text,
    re.IGNORECASE
)

# Q3 starts with 3. (i)
q3_start_match = re.search(
    r'3\.\s*\(i\)',
    text,
    re.IGNORECASE
)

if not q2_start_match:

    print("❌ Q2 could not be found in the PDF.")

else:

    q2_start = q2_start_match.start()

    if q3_start_match:
        q2_end = q3_start_match.start()
    else:
        q2_end = len(text)

    q2_text = text[q2_start:q2_end]


    # ========================================================
    # 4. EXTRACT Q2(i) AND Q2(ii)
    # ========================================================

    pattern = r'\((i|ii)\)\s*'

    matches = list(
        re.finditer(
            pattern,
            q2_text,
            re.IGNORECASE
        )
    )

    q2_questions = []

    for i, match in enumerate(matches):

        sub_question = f"({match.group(1)})"

        start = match.end()

        if i + 1 < len(matches):
            end = matches[i + 1].start()
        else:
            end = len(q2_text)

        question = q2_text[start:end]

        # Clean whitespace
        question = re.sub(
            r'\s+',
            ' ',
            question
        ).strip()

        q2_questions.append({
            "sub_question": sub_question,
            "question_text": question
        })


    # ========================================================
    # 5. FIND ANSWERS SECTION
    # ========================================================

    answers_match = re.search(
        r'Answers',
        text,
        re.IGNORECASE
    )

    q2_answers = []

    if answers_match:

        answers_text = text[answers_match.start():]

        # Find Q2 answer
        answer_q2_match = re.search(
            r'2\.\s*\(i\)',
            answers_text,
            re.IGNORECASE
        )

        # Q3 answer marks the end of Q2 answers
        answer_q3_match = re.search(
            r'3\.\s*\(i\)',
            answers_text,
            re.IGNORECASE
        )

        if answer_q2_match:

            answer_start = answer_q2_match.start()

            if answer_q3_match:
                answer_end = answer_q3_match.start()
            else:
                answer_end = len(answers_text)

            q2_answer_text = answers_text[
                answer_start:answer_end
            ]


            # =================================================
            # 6. EXTRACT Q2(i) AND Q2(ii) ANSWERS
            # =================================================

            answer_matches = list(
                re.finditer(
                    pattern,
                    q2_answer_text,
                    re.IGNORECASE
                )
            )

            for i, match in enumerate(answer_matches):

                sub_question = f"({match.group(1)})"

                start = match.end()

                if i + 1 < len(answer_matches):
                    end = answer_matches[i + 1].start()
                else:
                    end = len(q2_answer_text)

                answer = q2_answer_text[start:end]

                # Clean answer
                answer = re.sub(
                    r'\s+',
                    ' ',
                    answer
                ).strip()

                q2_answers.append({
                    "sub_question": sub_question,
                    "answer": answer
                })


    # ========================================================
    # 7. EXTRACTION CHECK
    # ========================================================

    print("=" * 80)
    print("Q2 EXTRACTION CHECK")
    print("=" * 80)

    print("Q2 questions found:", len(q2_questions))
    print("Q2 answers found:", len(q2_answers))


    # ========================================================
    # 8. MAP QUESTIONS WITH ANSWERS
    # ========================================================

    mapped_q2 = []

    for question in q2_questions:

        sub_question = question["sub_question"]

        matching_answer = ""

        for answer_item in q2_answers:

            if (
                answer_item["sub_question"].lower()
                == sub_question.lower()
            ):

                matching_answer = answer_item["answer"]

                break


        mapped_q2.append({

            "question_id":
                f"ENG25-P1-Q2-{sub_question.strip('()')}",

            "board":
                "ISC",

            "class":
                "Class XII",

            "subject":
                "English",

            "year":
                2025,

            "paper":
                "Paper 1 - English Language",

            "source_type":
                "Solved Paper",

            "parent_question":
                "Q2",

            "sub_question":
                sub_question,

            "question_type":
                "Writing",

            "marks":
                10,

            "question_text":
                question["question_text"],

            "answer":
                matching_answer,

            "source_file":
                pdf_path
        })


    # ========================================================
    # 9. CREATE DATAFRAME
    # ========================================================

    q2_df = pd.DataFrame(mapped_q2)


    # ========================================================
    # 10. DISPLAY RESULT
    # ========================================================

    print("\n" + "=" * 80)
    print("Q2 QUESTION-ANSWER MAPPING")
    print("=" * 80)

    print("\nTotal Q2 records:", len(q2_df))

    if len(q2_df) > 0:
        display(q2_df)
    else:
        print("❌ No Q2 records were created.")


    # ========================================================
    # 11. VALIDATE ANSWERS
    # ========================================================

    print("\n" + "=" * 80)
    print("VALIDATION")
    print("=" * 80)

    if "answer" in q2_df.columns:

        missing_answers = q2_df[
            q2_df["answer"].fillna("").str.strip() == ""
        ]

        if len(missing_answers) == 0:

            print(
                "✅ All Q2 questions have corresponding answers."
            )

        else:

            print(
                "⚠️ Missing answers:",
                len(missing_answers)
            )

            display(missing_answers)

    else:

        print("❌ Answer column was not created.")


    # ========================================================
    # 12. SAVE CSV
    # ========================================================

    output_file = "ISC_English_Paper1_2025_Q2.csv"

    q2_df.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig"
    )

    print("\nCSV saved as:", output_file)

In [ ]:
import fitz
import re
import pandas as pd

# ============================================================
# 1. OPEN PDF
# ============================================================

pdf_path = "English Paper 1 - ISC.pdf"

doc = fitz.open(pdf_path)

all_text = ""

for page in doc:
    all_text += page.get_text() + "\n"


# ============================================================
# 2. BASIC CLEANING
# ============================================================

text = all_text

# Remove page-number artifacts such as (16), (18)
text = re.sub(r'\(\s*\d+\s*\)', '', text)

# Normalize spaces
text = re.sub(r'[ \t]+', ' ', text)

# Normalize newlines
text = re.sub(r'\n+', '\n', text)


# ============================================================
# 3. FIND ANSWERS SECTION
# ============================================================

answers_match = re.search(
    r'Answers',
    text,
    re.IGNORECASE
)

if not answers_match:

    print("❌ Answers section could not be found.")

else:

    answers_text = text[answers_match.start():]


    # ========================================================
    # 4. FIND Q3 ANSWER SECTION
    # ========================================================

    q3_match = re.search(
        r'3\.\s*\(i\)',
        answers_text,
        re.IGNORECASE
    )

    q4_match = re.search(
        r'4\.\s*\(i\)',
        answers_text,
        re.IGNORECASE
    )

    if not q3_match:

        print("❌ Q3 answers could not be found.")

    else:

        q3_start = q3_match.start()

        if q4_match:
            q3_end = q4_match.start()
        else:
            q3_end = len(answers_text)

        q3_answer_text = answers_text[
            q3_start:q3_end
        ]


        # ====================================================
        # 5. DISPLAY RAW Q3 ANSWER SECTION
        # ====================================================

        print("=" * 80)
        print("RAW Q3 ANSWER SECTION")
        print("=" * 80)

        print(q3_answer_text)


        # ====================================================
        # 6. FIND Q3(i), Q3(ii), Q3(iii)
        # ====================================================

        section_i = re.search(
            r'3\.\s*\(i\)',
            q3_answer_text,
            re.IGNORECASE
        )

        section_ii = re.search(
            r'\(ii\)',
            q3_answer_text,
            re.IGNORECASE
        )

        section_iii = re.search(
            r'\(iii\)',
            q3_answer_text,
            re.IGNORECASE
        )


        q3_records = []


        # ====================================================
        # 7. Q3(i) — 5 ANSWERS (a) TO (e)
        # ====================================================

        if section_i:

            start = section_i.end()

            if section_ii:
                end = section_ii.start()
            else:
                end = len(q3_answer_text)

            q3_i_text = q3_answer_text[start:end]

            # Find (a), (b), (c), (d), (e)
            matches = list(
                re.finditer(
                    r'\(([a-e])\)\s*',
                    q3_i_text,
                    re.IGNORECASE
                )
            )

            for i, match in enumerate(matches):

                sub_question = f"({match.group(1)})"

                answer_start = match.end()

                if i + 1 < len(matches):
                    answer_end = matches[i + 1].start()
                else:
                    answer_end = len(q3_i_text)

                answer = q3_i_text[
                    answer_start:answer_end
                ]

                answer = re.sub(
                    r'\s+',
                    ' ',
                    answer
                ).strip()

                q3_records.append({
                    "section": "Q3(i)",
                    "sub_question": sub_question,
                    "answer": answer
                })


        # ====================================================
        # 8. Q3(ii) — 10 ANSWERS (a) TO (j)
        # ====================================================

        if section_ii:

            start = section_ii.end()

            if section_iii:
                end = section_iii.start()
            else:
                end = len(q3_answer_text)

            q3_ii_text = q3_answer_text[start:end]

            # Find (a) through (j)
            matches = list(
                re.finditer(
                    r'\(([a-j])\)\s*',
                    q3_ii_text,
                    re.IGNORECASE
                )
            )

            for i, match in enumerate(matches):

                sub_question = f"({match.group(1)})"

                answer_start = match.end()

                if i + 1 < len(matches):
                    answer_end = matches[i + 1].start()
                else:
                    answer_end = len(q3_ii_text)

                answer = q3_ii_text[
                    answer_start:answer_end
                ]

                answer = re.sub(
                    r'\s+',
                    ' ',
                    answer
                ).strip()

                q3_records.append({
                    "section": "Q3(ii)",
                    "sub_question": sub_question,
                    "answer": answer
                })


        # ====================================================
        # 9. Q3(iii) — 10 ANSWERS (1) TO (10)
        # ====================================================

        if section_iii:

            start = section_iii.end()

            q3_iii_text = q3_answer_text[start:]

            # Find numbered answers 1. to 10.
            matches = list(
                re.finditer(
                    r'(?<!\d)(10|[1-9])\.\s*',
                    q3_iii_text
                )
            )

            for i, match in enumerate(matches):

                number = match.group(1)

                answer_start = match.end()

                if i + 1 < len(matches):
                    answer_end = matches[i + 1].start()
                else:
                    answer_end = len(q3_iii_text)

                answer = q3_iii_text[
                    answer_start:answer_end
                ]

                answer = re.sub(
                    r'\s+',
                    ' ',
                    answer
                ).strip()

                q3_records.append({
                    "section": "Q3(iii)",
                    "sub_question": number,
                    "answer": answer
                })


        # ====================================================
        # 10. CREATE DATAFRAME
        # ====================================================

        q3_df = pd.DataFrame(q3_records)


        # ====================================================
        # 11. EXTRACTION CHECK
        # ====================================================

        print("\n" + "=" * 80)
        print("Q3 ANSWER EXTRACTION CHECK")
        print("=" * 80)

        print("Total Q3 answers found:", len(q3_df))

        print("\nAnswers by section:")

        for section in ["Q3(i)", "Q3(ii)", "Q3(iii)"]:

            count = len(
                q3_df[
                    q3_df["section"] == section
                ]
            )

            print(
                f"{section}: {count}"
            )


        # ====================================================
        # 12. DISPLAY DATA
        # ====================================================

        print("\n" + "=" * 80)
        print("Q3 ANSWER DATA")
        print("=" * 80)

        display(q3_df)


        # ====================================================
        # 13. VALIDATION
        # ====================================================

        print("\n" + "=" * 80)
        print("VALIDATION")
        print("=" * 80)

        expected = {
            "Q3(i)": 5,
            "Q3(ii)": 10,
            "Q3(iii)": 10
        }

        validation_passed = True

        for section, expected_count in expected.items():

            actual_count = len(
                q3_df[
                    q3_df["section"] == section
                ]
            )

            if actual_count == expected_count:

                print(
                    f"✅ {section}: "
                    f"{actual_count}/{expected_count}"
                )

            else:

                print(
                    f"⚠️ {section}: "
                    f"{actual_count}/{expected_count}"
                )

                validation_passed = False


        # ====================================================
        # 14. SAVE CSV
        # ====================================================

        output_file = (
            "ISC_English_Paper1_2025_Q3_Answers.csv"
        )

        q3_df.to_csv(
            output_file,
            index=False,
            encoding="utf-8-sig"
        )

        print(
            "\nCSV saved as:",
            output_file
        )


        # ====================================================
        # 15. FINAL STATUS
        # ====================================================

        if validation_passed:

            print(
                "\n✅ Q3 answer extraction completed successfully."
            )

        else:

            print(
                "\n⚠️ Q3 extraction needs verification."
            )

In [ ]:
import fitz
import re
import pandas as pd

# ============================================================
# 1. OPEN PDF
# ============================================================

pdf_path = "English Paper 1 - ISC.pdf"

doc = fitz.open(pdf_path)

all_text = ""

for page in doc:
    all_text += page.get_text() + "\n"


# ============================================================
# 2. BASIC CLEANING
# ============================================================

text = all_text

# Remove PDF page-number artifacts
text = re.sub(r'\(\s*\d+\s*\)', '', text)

# Normalize spaces
text = re.sub(r'[ \t]+', ' ', text)

# Normalize new lines
text = re.sub(r'\n+', '\n', text)


# ============================================================
# 3. FIND QUESTION 3
# ============================================================

q3_match = re.search(
    r'3\.\s*\(i\)',
    text,
    re.IGNORECASE
)

q4_match = re.search(
    r'4\.\s*\(i\)',
    text,
    re.IGNORECASE
)

if not q3_match:

    print("❌ Q3 could not be found.")

else:

    q3_start = q3_match.start()

    if q4_match:
        q3_end = q4_match.start()
    else:
        q3_end = len(text)

    q3_question_text = text[q3_start:q3_end]


    # ========================================================
    # 4. FIND ANSWERS SECTION
    # ========================================================

    answers_match = re.search(
        r'Answers',
        text,
        re.IGNORECASE
    )

    if not answers_match:

        print("❌ Answers section could not be found.")

    else:

        answers_text = text[answers_match.start():]

        answer_q3_match = re.search(
            r'3\.\s*\(i\)',
            answers_text,
            re.IGNORECASE
        )

        answer_q4_match = re.search(
            r'4\.\s*\(i\)',
            answers_text,
            re.IGNORECASE
        )

        if not answer_q3_match:

            print("❌ Q3 answers could not be found.")

        else:

            answer_start = answer_q3_match.start()

            if answer_q4_match:
                answer_end = answer_q4_match.start()
            else:
                answer_end = len(answers_text)

            q3_answer_text = answers_text[
                answer_start:answer_end
            ]


            # =================================================
            # 5. EXTRACT QUESTIONS
            # =================================================

            question_records = []


            # =================================================
            # Q3(i) QUESTIONS
            # =================================================

            q3_i_match = re.search(
                r'3\.\s*\(i\)',
                q3_question_text,
                re.IGNORECASE
            )

            q3_ii_match = re.search(
                r'\(ii\)',
                q3_question_text,
                re.IGNORECASE
            )

            q3_iii_match = re.search(
                r'\(iii\)',
                q3_question_text,
                re.IGNORECASE
            )


            if q3_i_match:

                start = q3_i_match.end()

                if q3_ii_match:
                    end = q3_ii_match.start()
                else:
                    end = len(q3_question_text)

                section = q3_question_text[start:end]

                matches = list(
                    re.finditer(
                        r'\(([a-e])\)\s*',
                        section,
                        re.IGNORECASE
                    )
                )

                for i, match in enumerate(matches):

                    sub_question = f"({match.group(1)})"

                    q_start = match.end()

                    if i + 1 < len(matches):
                        q_end = matches[i + 1].start()
                    else:
                        q_end = len(section)

                    question = section[q_start:q_end]

                    question = re.sub(
                        r'\s+',
                        ' ',
                        question
                    ).strip()

                    question_records.append({
                        "section": "Q3(i)",
                        "sub_question": sub_question,
                        "question_text": question
                    })


            # =================================================
            # Q3(ii) QUESTIONS
            # =================================================

            if q3_ii_match:

                start = q3_ii_match.end()

                if q3_iii_match:
                    end = q3_iii_match.start()
                else:
                    end = len(q3_question_text)

                section = q3_question_text[start:end]

                matches = list(
                    re.finditer(
                        r'\(([a-j])\)\s*',
                        section,
                        re.IGNORECASE
                    )
                )

                for i, match in enumerate(matches):

                    sub_question = f"({match.group(1)})"

                    q_start = match.end()

                    if i + 1 < len(matches):
                        q_end = matches[i + 1].start()
                    else:
                        q_end = len(section)

                    question = section[q_start:q_end]

                    question = re.sub(
                        r'\s+',
                        ' ',
                        question
                    ).strip()

                    question_records.append({
                        "section": "Q3(ii)",
                        "sub_question": sub_question,
                        "question_text": question
                    })


            # =================================================
            # Q3(iii) QUESTIONS
            # =================================================

            if q3_iii_match:

                start = q3_iii_match.end()

                section = q3_question_text[start:]

                matches = list(
                    re.finditer(
                        r'(?<!\d)(10|[1-9])\.\s*',
                        section
                    )
                )

                for i, match in enumerate(matches):

                    sub_question = match.group(1)

                    q_start = match.end()

                    if i + 1 < len(matches):
                        q_end = matches[i + 1].start()
                    else:
                        q_end = len(section)

                    question = section[q_start:q_end]

                    question = re.sub(
                        r'\s+',
                        ' ',
                        question
                    ).strip()

                    question_records.append({
                        "section": "Q3(iii)",
                        "sub_question": sub_question,
                        "question_text": question
                    })


            # =================================================
            # 6. EXTRACT ANSWERS
            # =================================================

            answer_records = []


            # =================================================
            # Q3(i) ANSWERS
            # =================================================

            answer_i_match = re.search(
                r'3\.\s*\(i\)',
                q3_answer_text,
                re.IGNORECASE
            )

            answer_ii_match = re.search(
                r'\(ii\)',
                q3_answer_text,
                re.IGNORECASE
            )

            answer_iii_match = re.search(
                r'\(iii\)',
                q3_answer_text,
                re.IGNORECASE
            )


            if answer_i_match:

                start = answer_i_match.end()

                if answer_ii_match:
                    end = answer_ii_match.start()
                else:
                    end = len(q3_answer_text)

                section = q3_answer_text[start:end]

                matches = list(
                    re.finditer(
                        r'\(([a-e])\)\s*',
                        section,
                        re.IGNORECASE
                    )
                )

                for i, match in enumerate(matches):

                    sub_question = f"({match.group(1)})"

                    a_start = match.end()

                    if i + 1 < len(matches):
                        a_end = matches[i + 1].start()
                    else:
                        a_end = len(section)

                    answer = section[a_start:a_end]

                    answer = re.sub(
                        r'\s+',
                        ' ',
                        answer
                    ).strip()

                    answer_records.append({
                        "section": "Q3(i)",
                        "sub_question": sub_question,
                        "answer": answer
                    })


            # =================================================
            # Q3(ii) ANSWERS
            # =================================================

            if answer_ii_match:

                start = answer_ii_match.end()

                if answer_iii_match:
                    end = answer_iii_match.start()
                else:
                    end = len(q3_answer_text)

                section = q3_answer_text[start:end]

                matches = list(
                    re.finditer(
                        r'\(([a-j])\)\s*',
                        section,
                        re.IGNORECASE
                    )
                )

                for i, match in enumerate(matches):

                    sub_question = f"({match.group(1)})"

                    a_start = match.end()

                    if i + 1 < len(matches):
                        a_end = matches[i + 1].start()
                    else:
                        a_end = len(section)

                    answer = section[a_start:a_end]

                    answer = re.sub(
                        r'\s+',
                        ' ',
                        answer
                    ).strip()

                    answer_records.append({
                        "section": "Q3(ii)",
                        "sub_question": sub_question,
                        "answer": answer
                    })


            # =================================================
            # Q3(iii) ANSWERS
            # =================================================

            if answer_iii_match:

                start = answer_iii_match.end()

                section = q3_answer_text[start:]

                matches = list(
                    re.finditer(
                        r'(?<!\d)(10|[1-9])\.\s*',
                        section
                    )
                )

                for i, match in enumerate(matches):

                    sub_question = match.group(1)

                    a_start = match.end()

                    if i + 1 < len(matches):
                        a_end = matches[i + 1].start()
                    else:
                        a_end = len(section)

                    answer = section[a_start:a_end]

                    answer = re.sub(
                        r'\s+',
                        ' ',
                        answer
                    ).strip()

                    answer_records.append({
                        "section": "Q3(iii)",
                        "sub_question": sub_question,
                        "answer": answer
                    })


            # =================================================
            # 7. MAP QUESTIONS WITH ANSWERS
            # =================================================

            mapped_records = []

            for question in question_records:

                section = question["section"]
                sub_question = question["sub_question"]

                matching_answer = ""

                for answer in answer_records:

                    if (
                        answer["section"] == section
                        and
                        answer["sub_question"].lower()
                        == sub_question.lower()
                    ):
                        matching_answer = answer["answer"]
                        break


                # Determine question type
                if section == "Q3(i)":
                    question_type = "Sentence Transformation"
                    marks = 5

                elif section == "Q3(ii)":
                    question_type = "Phrasal Verbs"
                    marks = 5

                else:
                    question_type = "Verb Forms"
                    marks = 10


                mapped_records.append({

                    "question_id":
                        f"ENG25-P1-Q3-{section[-4:-1]}-{sub_question.strip('()')}",

                    "board":
                        "ISC",

                    "class":
                        "Class XII",

                    "subject":
                        "English",

                    "year":
                        2025,

                    "paper":
                        "Paper 1 - English Language",

                    "source_type":
                        "Solved Paper",

                    "parent_question":
                        "Q3",

                    "section":
                        section,

                    "sub_question":
                        sub_question,

                    "question_type":
                        question_type,

                    "marks":
                        marks,

                    "question_text":
                        question["question_text"],

                    "answer":
                        matching_answer,

                    "source_file":
                        pdf_path
                })


            # =================================================
            # 8. CREATE DATAFRAME
            # =================================================

            q3_mapped_df = pd.DataFrame(
                mapped_records
            )


            # =================================================
            # 9. DISPLAY RESULT
            # =================================================

            print("=" * 80)
            print("Q3 QUESTION-ANSWER MAPPING")
            print("=" * 80)

            print(
                "\nTotal Q3 records:",
                len(q3_mapped_df)
            )

            display(q3_mapped_df)


            # =================================================
            # 10. VALIDATION
            # =================================================

            print("\n" + "=" * 80)
            print("VALIDATION")
            print("=" * 80)

            expected = {
                "Q3(i)": 5,
                "Q3(ii)": 10,
                "Q3(iii)": 10
            }

            validation_passed = True

            for section, expected_count in expected.items():

                actual_count = len(
                    q3_mapped_df[
                        q3_mapped_df["section"] == section
                    ]
                )

                missing_count = len(
                    q3_mapped_df[
                        (q3_mapped_df["section"] == section)
                        &
                        (
                            q3_mapped_df["answer"]
                            .fillna("")
                            .str.strip() == ""
                        )
                    ]
                )

                if (
                    actual_count == expected_count
                    and missing_count == 0
                ):

                    print(
                        f"✅ {section}: "
                        f"{actual_count}/{expected_count} "
                        f"mapped successfully"
                    )

                else:

                    print(
                        f"⚠️ {section}: "
                        f"{actual_count}/{expected_count}, "
                        f"missing answers: {missing_count}"
                    )

                    validation_passed = False


            # =================================================
            # 11. SAVE FINAL Q3 DATASET
            # =================================================

            output_file = (
                "ISC_English_Paper1_2025_Q3_Mapped.csv"
            )

            q3_mapped_df.to_csv(
                output_file,
                index=False,
                encoding="utf-8-sig"
            )

            print(
                "\nCSV saved as:",
                output_file
            )


            # =================================================
            # 12. FINAL STATUS
            # =================================================

            if validation_passed:

                print(
                    "\n✅ Q3 question-answer mapping completed successfully."
                )

            else:

                print(
                    "\n⚠️ Q3 mapping needs verification."
                )

In [ ]:
import fitz
import pandas as pd
import re

# ============================================================
# 1. FILE PATH
# ============================================================

pdf_path = "English Paper 1 - ISC.pdf"

doc = fitz.open(pdf_path)

# ============================================================
# 2. EXTRACT Q4 QUESTION TEXT
# ============================================================

q4_text = ""

# Q4 questions are on PDF pages 2 and 3
for page_num in [2, 3]:
    q4_text += doc[page_num - 1].get_text() + "\n"

# Clean PDF artifacts
q4_text = re.sub(r'\(\s*\d+\s*\)', '', q4_text)
q4_text = re.sub(r'\s+', ' ', q4_text).strip()

print("Q4 question text extracted successfully.")
print("=" * 80)
print(q4_text[:3000])

# ============================================================
# 3. CREATE Q4 RECORDS
# ============================================================

records = []

# ------------------------------------------------------------
# Q4(i)(a) - Vocabulary
# ------------------------------------------------------------

records.append({
    "question_id": "ENG25-P1-Q4-i-a-1",
    "board": "ISC",
    "class": "12",
    "subject": "English Language",
    "paper": "Paper 1",
    "year": 2025,
    "source_type": "Solved Board Paper",
    "question_number": "Q4(i)(a)(1)",
    "question_text": "The aircraft was designed to overcome sudden and violent changes in weather conditions.",
    "instruction": "Find a single word from the passage that will exactly replace the underlined word or words.",
    "question_type": "Vocabulary",
    "marks": 1,
    "answer": "turbulence",
    "source_file": "English Paper 1 - ISC.pdf"
})

records.append({
    "question_id": "ENG25-P1-Q4-i-a-2",
    "board": "ISC",
    "class": "12",
    "subject": "English Language",
    "paper": "Paper 1",
    "year": 2025,
    "source_type": "Solved Board Paper",
    "question_number": "Q4(i)(a)(2)",
    "question_text": "The doctor has advised the patient to continue taking the medicines till the fever comes down.",
    "instruction": "Find a single word from the passage that will exactly replace the underlined word or words.",
    "question_type": "Vocabulary",
    "marks": 1,
    "answer": "subsides",
    "source_file": "English Paper 1 - ISC.pdf"
})

records.append({
    "question_id": "ENG25-P1-Q4-i-a-3",
    "board": "ISC",
    "class": "12",
    "subject": "English Language",
    "paper": "Paper 1",
    "year": 2025,
    "source_type": "Solved Board Paper",
    "question_number": "Q4(i)(a)(3)",
    "question_text": "The thief escapes the police every time.",
    "instruction": "Find a single word from the passage that will exactly replace the underlined word or words.",
    "question_type": "Vocabulary",
    "marks": 1,
    "answer": "eludes",
    "source_file": "English Paper 1 - ISC.pdf"
})

# ------------------------------------------------------------
# Q4(i)(b) - Same word, different meaning
# ------------------------------------------------------------

records.append({
    "question_id": "ENG25-P1-Q4-i-b-1",
    "board": "ISC",
    "class": "12",
    "subject": "English Language",
    "paper": "Paper 1",
    "year": 2025,
    "source_type": "Solved Board Paper",
    "question_number": "Q4(i)(b)(1)",
    "question_text": "steps (line 2)",
    "instruction": "Choose the correct sentence that uses the same word unchanged in spelling, but with a different meaning from that which it carries in the passage.",
    "question_type": "MCQ",
    "marks": 1,
    "answer": "A",
    "source_file": "English Paper 1 - ISC.pdf"
})

records.append({
    "question_id": "ENG25-P1-Q4-i-b-2",
    "board": "ISC",
    "class": "12",
    "subject": "English Language",
    "paper": "Paper 1",
    "year": 2025,
    "source_type": "Solved Board Paper",
    "question_number": "Q4(i)(b)(2)",
    "question_text": "light (line 44)",
    "instruction": "Choose the correct sentence that uses the same word unchanged in spelling, but with a different meaning from that which it carries in the passage.",
    "question_type": "MCQ",
    "marks": 1,
    "answer": "C",
    "source_file": "English Paper 1 - ISC.pdf"
})

records.append({
    "question_id": "ENG25-P1-Q4-i-b-3",
    "board": "ISC",
    "class": "12",
    "subject": "English Language",
    "paper": "Paper 1",
    "year": 2025,
    "source_type": "Solved Board Paper",
    "question_number": "Q4(i)(b)(3)",
    "question_text": "set (line 44)",
    "instruction": "Choose the correct sentence that uses the same word unchanged in spelling, but with a different meaning from that which it carries in the passage.",
    "question_type": "MCQ",
    "marks": 1,
    "answer": "A",
    "source_file": "English Paper 1 - ISC.pdf"
})

# ------------------------------------------------------------
# Q4(ii) - Short Answers
# ------------------------------------------------------------

records.append({
    "question_id": "ENG25-P1-Q4-ii-a",
    "board": "ISC",
    "class": "12",
    "subject": "English Language",
    "paper": "Paper 1",
    "year": 2025,
    "source_type": "Solved Board Paper",
    "question_number": "Q4(ii)(a)",
    "question_text": "Why was the expedition important to Emma?",
    "instruction": "Answer the following questions as briefly as possible in your own words.",
    "question_type": "Short Answer",
    "marks": 2,
    "answer": "The expedition was important to Emma because she aimed to be the first scientist to explore inside Mount Michael’s crater, collect clues about active volcanoes, and find the lava lake.",
    "source_file": "English Paper 1 - ISC.pdf"
})

records.append({
    "question_id": "ENG25-P1-Q4-ii-b",
    "board": "ISC",
    "class": "12",
    "subject": "English Language",
    "paper": "Paper 1",
    "year": 2025,
    "source_type": "Solved Board Paper",
    "question_number": "Q4(ii)(b)",
    "question_text": "With reference to the role of the mountain guide in the passage, explain how trust plays an important role in Emma’s safety.",
    "instruction": "Answer the following questions as briefly as possible in your own words.",
    "question_type": "Short Answer",
    "marks": 2,
    "answer": "Carla, the mountain guide, is Emma’s anchor and ensures her safety. Trusting Carla’s expertise is crucial, as Carla’s vigilance in monitoring the rope and surrounding conditions protects Emma from potential dangers like falling ice cliffs.",
    "source_file": "English Paper 1 - ISC.pdf"
})

records.append({
    "question_id": "ENG25-P1-Q4-ii-c",
    "board": "ISC",
    "class": "12",
    "subject": "English Language",
    "paper": "Paper 1",
    "year": 2025,
    "source_type": "Solved Board Paper",
    "question_number": "Q4(ii)(c)",
    "question_text": "Why is Mount Michael considered a remote place?",
    "instruction": "Answer the following questions as briefly as possible in your own words.",
    "question_type": "Short Answer",
    "marks": 2,
    "answer": "Mount Michael is considered remote due to its isolated location in the uninhabited South Sandwich archipelago, far from the nearest permanent station and shipping traffic.",
    "source_file": "English Paper 1 - ISC.pdf"
})

# ------------------------------------------------------------
# Q4(iii) - Summary
# ------------------------------------------------------------

records.append({
    "question_id": "ENG25-P1-Q4-iii",
    "board": "ISC",
    "class": "12",
    "subject": "English Language",
    "paper": "Paper 1",
    "year": 2025,
    "source_type": "Solved Board Paper",
    "question_number": "Q4(iii)",
    "question_text": "Summarise Emma’s search for the lava lake and her joy at witnessing it finally (paragraphs 8 to 14). You are required to write the summary in the form of a connected passage in about 100 words.",
    "instruction": "Write the summary in the form of a connected passage in about 100 words.",
    "question_type": "Summary",
    "marks": 8,
    "answer": "Emma’s arduous journey to Mount Michael, driven by her quest to find its elusive lava lake, leads her into the volcano’s depths. She cautiously navigates icy slopes and ash-covered rock, aware of the hidden dangers. The vast crater showcases nature’s raw power, yet after sixteen days, the lava lake remains undiscovered. On their final night, as the team prepares to leave, a shimmering cloud appears above the volcano, glowing in shifting hues of red, orange and purple. Emma realises the light is coming from within—confirming the lava lake’s presence. Overcome with excitement, she knows her perseverance has finally been rewarded.",
    "source_file": "English Paper 1 - ISC.pdf"
})

# ============================================================
# 4. CREATE DATAFRAME
# ============================================================

df_q4 = pd.DataFrame(records)

# ============================================================
# 5. VALIDATE
# ============================================================

print("\n")
print("=" * 80)
print("Q4 DATASET VALIDATION")
print("=" * 80)

print("Total Q4 records:", len(df_q4))
print("Total Q4 marks:", df_q4["marks"].sum())

print("\nRecords by question type:")
print(df_q4["question_type"].value_counts())

print("\nQ4 QUESTIONS AND ANSWERS:")
print("=" * 80)

display(df_q4[[
    "question_id",
    "question_number",
    "question_type",
    "marks",
    "question_text",
    "answer"
]])

# ============================================================
# 6. SAVE CSV
# ============================================================

output_file = "ISC_Class12_English_Paper1_2025_Q4.csv"

df_q4.to_csv(
    output_file,
    index=False,
    encoding="utf-8"
)

print("\nCSV saved successfully as:", output_file)

In [ ]:
import fitz
import pandas as pd

# ============================================================
# 1. FILE
# ============================================================

pdf_path = "English Paper 1 - ISC.pdf"

doc = fitz.open(pdf_path)

# ============================================================
# 2. CREATE Q3 RECORDS
# ============================================================

records = []

# ============================================================
# Q3(i) - SENTENCE TRANSFORMATION
# ============================================================

q3_i = [
    {
        "subpart": "a",
        "question_text": "As soon as Mira found her lost diamond pendant, she ran to her mother in excitement. (Begin: No sooner ....................)",
        "answer": "No sooner did Mira find her lost diamond pendant than she ran to her mother in excitement."
    },
    {
        "subpart": "b",
        "question_text": "The doctor said to the elderly lady, “Will you be able to follow the schedule for taking medicines every day?” (Begin: The doctor asked ...................)",
        "answer": "The doctor asked the elderly lady if she would be able to follow the schedule for taking medicines every day."
    },
    {
        "subpart": "c",
        "question_text": "Heavy rains have caused several landslides. (End: ......................... heavy rains.)",
        "answer": "Several landslides have been caused by heavy rains."
    },
    {
        "subpart": "d",
        "question_text": "Sunita is the best debater in our team. (Begin: No other .................................)",
        "answer": "No other debater in our team is as good as Sunita."
    },
    {
        "subpart": "e",
        "question_text": "The news is too good to be true. (Use: .......... so .......... that ................)",
        "answer": "The news is so good that it cannot be true."
    }
]

for item in q3_i:
    records.append({
        "question_id": f"ENG25-P1-Q3-i-{item['subpart']}",
        "board": "ISC",
        "class": "12",
        "subject": "English Language",
        "paper": "Paper 1",
        "year": 2025,
        "source_type": "Solved Board Paper",
        "question_number": f"Q3(i)({item['subpart']})",
        "question_text": item["question_text"],
        "instruction": "Rewrite the following sentence according to the instruction given in brackets.",
        "question_type": "Sentence Transformation",
        "marks": 1,
        "answer": item["answer"],
        "source_file": "English Paper 1 - ISC.pdf"
    })


# ============================================================
# Q3(ii) - PHRASAL VERBS
# ============================================================

q3_ii = [
    {
        "subpart": "a",
        "question_text": "Never put out until tomorrow what you can do today.",
        "answer": "off"
    },
    {
        "subpart": "b",
        "question_text": "Put by your best dress. We are going to attend a party.",
        "answer": "on"
    },
    {
        "subpart": "c",
        "question_text": "She is so friendly that she gets away with everyone.",
        "answer": "along"
    },
    {
        "subpart": "d",
        "question_text": "I must get up to writing this assignment as it is due tomorrow.",
        "answer": "down"
    },
    {
        "subpart": "e",
        "question_text": "The engagement was called through at the last moment.",
        "answer": "off"
    },
    {
        "subpart": "f",
        "question_text": "This situation calls about necessary action.",
        "answer": "for"
    },
    {
        "subpart": "g",
        "question_text": "Saloni chose to stand up her friend in times of crisis.",
        "answer": "by"
    },
    {
        "subpart": "h",
        "question_text": "IPL stands of Indian Premier League.",
        "answer": "for"
    },
    {
        "subpart": "i",
        "question_text": "Neha turned out the job offer as she wanted to continue her higher studies.",
        "answer": "down"
    },
    {
        "subpart": "j",
        "question_text": "I did not expect so many people to turn over for the conference.",
        "answer": "out"
    }
]

for item in q3_ii:
    records.append({
        "question_id": f"ENG25-P1-Q3-ii-{item['subpart']}",
        "board": "ISC",
        "class": "12",
        "subject": "English Language",
        "paper": "Paper 1",
        "year": 2025,
        "source_type": "Solved Board Paper",
        "question_number": f"Q3(ii)({item['subpart']})",
        "question_text": item["question_text"],
        "instruction": "Replace the INCORRECT underlined word to make the sentence meaningful and grammatically correct. Do not copy the sentence.",
        "question_type": "Phrasal Verb",
        "marks": 1,
        "answer": item["answer"],
        "source_file": "English Paper 1 - ISC.pdf"
    })


# ============================================================
# Q3(iii) - VERB FORMS
# ============================================================

q3_iii = [
    {
        "number": 1,
        "question_text": "Once upon a time, there ______ (live) three old monks.",
        "answer": "lived"
    },
    {
        "number": 2,
        "question_text": "They ______ (be) poor and without any worldly goods.",
        "answer": "were"
    },
    {
        "number": 3,
        "question_text": "They would travel from village to village and, in every village they ______ (stop), they would sit in the center of the marketplace and start laughing.",
        "answer": "stopped"
    },
    {
        "number": 4,
        "question_text": "At first, everyone just ______ (pass) by without giving them any attention.",
        "answer": "passed"
    },
    {
        "number": 5,
        "question_text": "This was when the monks would finally get up and ______ (move) on to the next village.",
        "answer": "move"
    },
    {
        "number": 6,
        "question_text": "Their fame ______ (grow) with time.",
        "answer": "grew"
    },
    {
        "number": 7,
        "question_text": "Nobody ever ______ (preach) the way they did with laughter and nothing else.",
        "answer": "preached"
    },
    {
        "number": 8,
        "question_text": "They never laughed at anything in particular, but it was as if they ______ (understand) a cosmic joke.",
        "answer": "understood"
    },
    {
        "number": 9,
        "question_text": "Just ______ (imagine)!",
        "answer": "imagine"
    },
    {
        "number": 10,
        "question_text": "Three monks ______ (spread) joy and laughter without saying a single word.",
        "answer": "spreading"
    }
]

for item in q3_iii:
    records.append({
        "question_id": f"ENG25-P1-Q3-iii-{item['number']}",
        "board": "ISC",
        "class": "12",
        "subject": "English Language",
        "paper": "Paper 1",
        "year": 2025,
        "source_type": "Solved Board Paper",
        "question_number": f"Q3(iii)({item['number']})",
        "question_text": item["question_text"],
        "instruction": "Fill in the blank with the appropriate form of the verb given in brackets.",
        "question_type": "Verb Form",
        "marks": 1,
        "answer": item["answer"],
        "source_file": "English Paper 1 - ISC.pdf"
    })


# ============================================================
# 3. CREATE DATAFRAME
# ============================================================

df_q3 = pd.DataFrame(records)


# ============================================================
# 4. VALIDATION
# ============================================================

print("=" * 80)
print("Q3 DATASET VALIDATION")
print("=" * 80)

print("Total Q3 records:", len(df_q3))
print("Total Q3 marks:", df_q3["marks"].sum())

print("\nRecords by question type:")
print(df_q3["question_type"].value_counts())

print("\nRecords by section:")
print(df_q3["question_number"].str.extract(r"(Q3\(i{1,3}\))")[0].value_counts())


# ============================================================
# 5. DISPLAY QUESTIONS AND ANSWERS
# ============================================================

print("\n" + "=" * 80)
print("Q3 QUESTIONS AND ANSWERS")
print("=" * 80)

display(
    df_q3[
        [
            "question_id",
            "question_number",
            "question_type",
            "marks",
            "question_text",
            "answer"
        ]
    ]
)


# ============================================================
# 6. SAVE CSV
# ============================================================

output_file = "ISC_Class12_English_Paper1_2025_Q3.csv"

df_q3.to_csv(
    output_file,
    index=False,
    encoding="utf-8"
)

print("\nCSV saved successfully as:", output_file)

In [ ]:
import os
import glob

print("CSV files currently available in Colab:\n")

csv_files = glob.glob("*.csv")

if csv_files:
    for file in csv_files:
        print(file)
else:
    print("No CSV files found.")

In [ ]:
import pandas as pd
import os

# ============================================================
# 1. CORRECT FILES
# ============================================================

q1_file = "ISC_English_Paper1_2025_Q1.csv"
q2_file = "ISC_English_Paper1_2025_Q2.csv"
q3_file = "ISC_Class12_English_Paper1_2025_Q3.csv"
q4_file = "ISC_Class12_English_Paper1_2025_Q4.csv"

files = [q1_file, q2_file, q3_file, q4_file]

# ============================================================
# 2. CHECK FILES
# ============================================================

print("=" * 80)
print("CHECKING FILES")
print("=" * 80)

for file in files:
    if os.path.exists(file):
        print("✓", file)
    else:
        print("✗", file, "NOT FOUND")
        raise FileNotFoundError(f"{file} is missing.")

# ============================================================
# 3. LOAD FILES
# ============================================================

df_q1 = pd.read_csv(q1_file)
df_q2 = pd.read_csv(q2_file)
df_q3 = pd.read_csv(q3_file)
df_q4 = pd.read_csv(q4_file)

print("\n")
print("=" * 80)
print("RECORD COUNTS")
print("=" * 80)

print("Q1:", len(df_q1))
print("Q2:", len(df_q2))
print("Q3:", len(df_q3))
print("Q4:", len(df_q4))

# ============================================================
# 4. COMBINE IN QUESTION ORDER
# ============================================================

df_master = pd.concat(
    [df_q1, df_q2, df_q3, df_q4],
    ignore_index=True
)

# ============================================================
# 5. BASIC CLEANING
# ============================================================

df_master.reset_index(drop=True, inplace=True)

# ============================================================
# 6. VALIDATE REQUIRED COLUMNS
# ============================================================

required_columns = [
    "question_id",
    "board",
    "class",
    "subject",
    "paper",
    "year",
    "source_type",
    "question_number",
    "question_text",
    "instruction",
    "question_type",
    "marks",
    "answer",
    "source_file"
]

print("\n")
print("=" * 80)
print("COLUMN VALIDATION")
print("=" * 80)

missing_columns = [
    col for col in required_columns
    if col not in df_master.columns
]

if missing_columns:
    print("✗ Missing columns:")
    for col in missing_columns:
        print("-", col)
else:
    print("✓ All required columns are present.")

# ============================================================
# 7. RECORD VALIDATION
# ============================================================

print("\n")
print("=" * 80)
print("RECORD VALIDATION")
print("=" * 80)

print("Total records:", len(df_master))

expected_records = 43

if len(df_master) == expected_records:
    print(f"✓ Record count is correct: {expected_records}")
else:
    print(f"⚠ Expected {expected_records} records, but found {len(df_master)}")

# ============================================================
# 8. DUPLICATE QUESTION ID CHECK
# ============================================================

duplicate_ids = df_master[
    df_master["question_id"].duplicated(keep=False)
]

print("\nDuplicate question IDs:", len(duplicate_ids))

if len(duplicate_ids) == 0:
    print("✓ No duplicate question IDs.")
else:
    print("⚠ Duplicate IDs found:")
    display(duplicate_ids[["question_id", "question_number"]])

# ============================================================
# 9. MISSING QUESTION / ANSWER CHECK
# ============================================================

empty_questions = df_master[
    df_master["question_text"].isna() |
    (df_master["question_text"].astype(str).str.strip() == "")
]

empty_answers = df_master[
    df_master["answer"].isna() |
    (df_master["answer"].astype(str).str.strip() == "")
]

print("\nEmpty question_text records:", len(empty_questions))
print("Empty answer records:", len(empty_answers))

if len(empty_questions) == 0:
    print("✓ No empty questions.")
else:
    print("⚠ Empty questions found:")
    display(empty_questions)

if len(empty_answers) == 0:
    print("✓ No empty answers.")
else:
    print("⚠ Empty answers found:")
    display(empty_answers)

# ============================================================
# 10. QUESTION TYPE DISTRIBUTION
# ============================================================

print("\n")
print("=" * 80)
print("QUESTION TYPE DISTRIBUTION")
print("=" * 80)

print(
    df_master["question_type"]
    .value_counts()
)

# ============================================================
# 11. SECTION DISTRIBUTION
# ============================================================

print("\n")
print("=" * 80)
print("RECORDS BY QUESTION")
print("=" * 80)

question_distribution = (
    df_master["question_number"]
    .astype(str)
    .str.extract(r"(Q[1-4])", expand=False)
    .value_counts()
    .sort_index()
)

print(question_distribution)

# ============================================================
# 12. DATASET MARKS
# ============================================================

print("\n")
print("=" * 80)
print("MARK INFORMATION")
print("=" * 80)

print("Sum of record-level marks:", df_master["marks"].sum())

print("\nMarks by question:")
print(
    df_master.groupby(
        df_master["question_number"]
        .astype(str)
        .str.extract(r"(Q[1-4])", expand=False)
    )["marks"].sum()
)

# ============================================================
# IMPORTANT:
# The record-level sum should NOT necessarily equal 80.
#
# Q1 contains 6 alternative composition questions.
# A student answers only ONE of them.
#
# Therefore, Q1's paper-level contribution = 20 marks,
# not 6 x 20 marks.
#
# Q3 also contains multiple individual items within
# sections carrying a combined mark allocation.
# ============================================================

print("\n")
print("=" * 80)
print("PAPER-LEVEL STRUCTURE")
print("=" * 80)

print("Q1 = 20 marks (student chooses ONE composition)")
print("Q2 = 25 marks")
print("Q3 = 15 marks")
print("Q4 = 20 marks")
print("-" * 40)
print("Paper maximum = 80 marks")

# ============================================================
# 13. SHOW MASTER DATASET
# ============================================================

print("\n")
print("=" * 80)
print("MASTER DATASET PREVIEW")
print("=" * 80)

display(df_master)

# ============================================================
# 14. SAVE MASTER CSV
# ============================================================

master_file = "ISC_Class12_English_Paper1_2025_MASTER.csv"

df_master.to_csv(
    master_file,
    index=False,
    encoding="utf-8"
)

print("\n")
print("=" * 80)
print("SUCCESS")
print("=" * 80)

print(f"✓ Master CSV saved successfully:")
print(master_file)

In [ ]:
import pandas as pd
import os

# ============================================================
# 1. LOAD MASTER CSV
# ============================================================

master_file = "ISC_Class12_English_Paper1_2025_MASTER.csv"

if not os.path.exists(master_file):
    raise FileNotFoundError(
        f"{master_file} was not found. Please make sure the master CSV "
        "is present in your Colab environment."
    )

df = pd.read_csv(master_file)

print("=" * 80)
print("ISC CLASS 12 ENGLISH PAPER 1 — FINAL QUALITY CHECK")
print("=" * 80)

print("\nTotal records:", len(df))
print("Total columns:", len(df.columns))

# ============================================================
# 2. COLUMN CHECK
# ============================================================

print("\n")
print("=" * 80)
print("1. COLUMN CHECK")
print("=" * 80)

print("Columns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

# ============================================================
# 3. REQUIRED COLUMN CHECK
# ============================================================

required_columns = [
    "question_id",
    "board",
    "class",
    "subject",
    "paper",
    "year",
    "source_type",
    "question_number",
    "question_text",
    "instruction",
    "question_type",
    "marks",
    "answer",
    "source_file"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

print("\n")
print("=" * 80)
print("2. REQUIRED COLUMN VALIDATION")
print("=" * 80)

if not missing_columns:
    print("✓ All required columns are present.")
else:
    print("✗ Missing columns:")
    for col in missing_columns:
        print("-", col)

# ============================================================
# 4. DATA TYPES
# ============================================================

print("\n")
print("=" * 80)
print("3. DATA TYPES")
print("=" * 80)

print(df.dtypes)

# ============================================================
# 5. MISSING VALUES
# ============================================================

print("\n")
print("=" * 80)
print("4. MISSING VALUE CHECK")
print("=" * 80)

missing = df.isnull().sum()

if missing.sum() == 0:
    print("✓ No NULL values found.")
else:
    print("Missing values by column:")
    print(missing[missing > 0])

# Also check empty strings
print("\nEmpty-string checks:")

for col in ["question_text", "answer", "instruction"]:
    if col in df.columns:
        empty_count = (
            df[col]
            .fillna("")
            .astype(str)
            .str.strip()
            .eq("")
            .sum()
        )
        print(f"{col}: {empty_count}")

# ============================================================
# 6. QUESTION ID CHECK
# ============================================================

print("\n")
print("=" * 80)
print("5. QUESTION ID CHECK")
print("=" * 80)

duplicate_ids = df[df["question_id"].duplicated(keep=False)]

print("Total question IDs:", df["question_id"].nunique())
print("Duplicate ID records:", len(duplicate_ids))

if len(duplicate_ids) == 0:
    print("✓ No duplicate question IDs.")
else:
    print("✗ Duplicate question IDs found:")
    display(
        duplicate_ids[
            ["question_id", "question_number", "question_text"]
        ]
    )

# ============================================================
# 7. QUESTION TEXT DUPLICATE CHECK
# ============================================================

print("\n")
print("=" * 80)
print("6. QUESTION TEXT DUPLICATE CHECK")
print("=" * 80)

duplicate_questions = df[
    df["question_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .duplicated(keep=False)
]

duplicate_question_df = df[duplicate_questions]

print("Potential exact duplicate question records:",
      len(duplicate_question_df))

if len(duplicate_question_df) > 0:
    display(
        duplicate_question_df[
            [
                "question_id",
                "question_number",
                "question_type",
                "question_text"
            ]
        ]
    )
else:
    print("✓ No exact duplicate question texts found.")

# ============================================================
# 8. QUESTION TYPE DISTRIBUTION
# ============================================================

print("\n")
print("=" * 80)
print("7. QUESTION TYPE DISTRIBUTION")
print("=" * 80)

print(
    df["question_type"]
    .value_counts(dropna=False)
)

# ============================================================
# 9. QUESTION DISTRIBUTION
# ============================================================

print("\n")
print("=" * 80)
print("8. QUESTION DISTRIBUTION")
print("=" * 80)

print(
    df["question_number"]
    .value_counts()
    .sort_index()
)

# ============================================================
# 10. MARKS CHECK
# ============================================================

print("\n")
print("=" * 80)
print("9. MARKS CHECK")
print("=" * 80)

print("Marks data type:", df["marks"].dtype)

print("\nMarks distribution:")
print(df["marks"].value_counts().sort_index())

print("\nRecord-level marks total:", df["marks"].sum())

# ============================================================
# 11. PAPER-LEVEL MARK STRUCTURE
# ============================================================

print("\n")
print("=" * 80)
print("10. PAPER-LEVEL MARK STRUCTURE")
print("=" * 80)

print("""
Q1 = 20 marks
     → 6 alternative composition topics
     → Student answers only ONE

Q2 = 25 marks
     → Feature Article = 15
     → Proposal = 10

Q3 = 15 marks
     → Sentence Transformation = 5
     → Phrasal Verb = 5
     → Verb Form = 5

Q4 = 20 marks

Paper Maximum = 80 marks
""")

# ============================================================
# 12. METADATA CONSISTENCY
# ============================================================

print("\n")
print("=" * 80)
print("11. METADATA CONSISTENCY")
print("=" * 80)

for col in ["board", "class", "subject", "paper", "year"]:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts(dropna=False))

# ============================================================
# 13. SOURCE TYPE CHECK
# ============================================================

print("\n")
print("=" * 80)
print("12. SOURCE TYPE CHECK")
print("=" * 80)

print(df["source_type"].value_counts(dropna=False))

# ============================================================
# 14. SOURCE FILE CHECK
# ============================================================

print("\n")
print("=" * 80)
print("13. SOURCE FILE CHECK")
print("=" * 80)

print(df["source_file"].value_counts(dropna=False))

# ============================================================
# 15. ANSWER LENGTH CHECK
# ============================================================

print("\n")
print("=" * 80)
print("14. ANSWER LENGTH CHECK")
print("=" * 80)

df["answer_length"] = (
    df["answer"]
    .fillna("")
    .astype(str)
    .str.len()
)

print(df["answer_length"].describe())

# ============================================================
# 16. QUESTION TEXT LENGTH CHECK
# ============================================================

print("\n")
print("=" * 80)
print("15. QUESTION TEXT LENGTH CHECK")
print("=" * 80)

df["question_length"] = (
    df["question_text"]
    .fillna("")
    .astype(str)
    .str.len()
)

print(df["question_length"].describe())

# ============================================================
# 17. DISPLAY SAMPLE RECORDS
# ============================================================

print("\n")
print("=" * 80)
print("16. SAMPLE RECORDS")
print("=" * 80)

display(
    df[
        [
            "question_id",
            "question_number",
            "question_type",
            "marks",
            "question_text",
            "answer"
        ]
    ].head(10)
)

# ============================================================
# 18. FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 80)
print("FINAL QUALITY CHECK SUMMARY")
print("=" * 80)

checks = {
    "43 records": len(df) == 43,
    "Required columns present": len(missing_columns) == 0,
    "No duplicate question IDs": len(duplicate_ids) == 0,
    "No NULL values": missing.sum() == 0,
    "Question text available": (
        df["question_text"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .all()
    ),
    "Answers available": (
        df["answer"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
        .all()
    )
}

for check, result in checks.items():
    print(("✓ " if result else "✗ ") + check)

# ============================================================
# 19. SAVE CLEAN VERSION
# ============================================================

# Remove temporary diagnostic columns before saving
df_clean = df.drop(
    columns=["answer_length", "question_length"]
)

clean_file = "ISC_Class12_English_Paper1_2025_MASTER_CLEAN.csv"

df_clean.to_csv(
    clean_file,
    index=False,
    encoding="utf-8"
)

print("\n")
print("=" * 80)
print("CLEAN MASTER FILE")
print("=" * 80)

print(f"✓ Saved as: {clean_file}")
print(f"✓ Records: {len(df_clean)}")
print(f"✓ Columns: {len(df_clean.columns)}")

In [ ]:
# ============================================================
# ISC CLASS 12 ENGLISH PAPER 1 — DATA CORRECTION
# ============================================================

import pandas as pd
import os
import glob
import re

# ============================================================
# 1. FILE NAMES
# ============================================================

master_file = "ISC_Class12_English_Paper1_2025_MASTER.csv"

# Try the known PDF name first
pdf_candidates = [
    "English Paper 1 - ISC.pdf",
    "/mnt/data/English Paper 1 - ISC.pdf"
]

# ============================================================
# 2. CHECK MASTER CSV
# ============================================================

if not os.path.exists(master_file):
    raise FileNotFoundError(
        f"{master_file} was not found."
    )

df = pd.read_csv(master_file)

print("=" * 80)
print("LOADED MASTER DATASET")
print("=" * 80)
print("Records:", len(df))
print("Columns:", len(df.columns))

# ============================================================
# 3. FIND ORIGINAL PDF
# ============================================================

pdf_file = None

for candidate in pdf_candidates:
    if os.path.exists(candidate):
        pdf_file = candidate
        break

# Also search current directory and /mnt/data
if pdf_file is None:
    all_pdfs = glob.glob("*.pdf") + glob.glob("/mnt/data/*.pdf")

    for p in all_pdfs:
        if "English" in os.path.basename(p) and "Paper" in os.path.basename(p):
            pdf_file = p
            break

if pdf_file is None:
    raise FileNotFoundError(
        "Original English Paper 1 PDF was not found in Colab.\n"
        "Please upload 'English Paper 1 - ISC.pdf' and run this cell again."
    )

print("PDF found:", pdf_file)

# ============================================================
# 4. EXTRACT PDF TEXT
# ============================================================

try:
    import fitz
except ImportError:
    !pip -q install pymupdf
    import fitz

doc = fitz.open(pdf_file)

pages = []

for page in doc:
    pages.append(page.get_text())

full_text = "\n".join(pages)

print("PDF pages:", len(doc))
print("PDF text extracted successfully.")

# ============================================================
# 5. SHOW WHERE Q1 / Q2 OCCUR
# ============================================================

print("\n")
print("=" * 80)
print("LOCATING Q1 AND Q2")
print("=" * 80)

q1_position = full_text.find("1.")
q2_position = full_text.find("2.")

print("Q1 position:", q1_position)
print("Q2 position:", q2_position)

if q1_position == -1 or q2_position == -1:
    raise ValueError(
        "Could not reliably locate Q1 and Q2 in the PDF."
    )

# ============================================================
# 6. PRINT PDF SECTION FOR MANUAL VERIFICATION
# ============================================================

print("\n")
print("=" * 80)
print("ORIGINAL PDF — Q1/Q2 SECTION")
print("=" * 80)

# Display enough text to inspect the original questions.
# We do NOT automatically use this entire block as question text.
preview_end = min(q2_position + 5000, len(full_text))

print(full_text[q1_position:preview_end])

# ============================================================
# 7. CORRECT MARKS
# ============================================================

print("\n")
print("=" * 80)
print("CORRECTING MARKS")
print("=" * 80)

# Keep a copy so we can report changes
before_marks = df["marks"].copy()

# Q2(i) = Feature Article = 15 marks
mask_q2_feature = (
    df["question_id"].astype(str).str.upper()
    == "ENG25-P1-Q2-I"
)

if mask_q2_feature.any():
    df.loc[mask_q2_feature, "marks"] = 15
    print("✓ Q2(i) Feature Article: 10 → 15")
else:
    print("⚠ Q2(i) ID not found.")

# Q2(ii) = Proposal = 10 marks
mask_q2_proposal = (
    df["question_id"].astype(str).str.upper()
    == "ENG25-P1-Q2-II"
)

if mask_q2_proposal.any():
    df.loc[mask_q2_proposal, "marks"] = 10
    print("✓ Q2(ii) Proposal: 10 → 10")

# Q3(i) = 5 items × 1 mark
mask_q3_i = (
    df["question_number"]
    .fillna("")
    .astype(str)
    .str.startswith("Q3(i)")
)

df.loc[mask_q3_i, "marks"] = 1

print(
    "✓ Q3(i):",
    mask_q3_i.sum(),
    "records × 1 mark"
)

# Q3(ii) = 10 items × 0.5 mark
mask_q3_ii = (
    df["question_number"]
    .fillna("")
    .astype(str)
    .str.startswith("Q3(ii)")
)

df.loc[mask_q3_ii, "marks"] = 0.5

print(
    "✓ Q3(ii):",
    mask_q3_ii.sum(),
    "records × 0.5 mark"
)

# Q3(iii) = 10 items × 0.5 mark
mask_q3_iii = (
    df["question_number"]
    .fillna("")
    .astype(str)
    .str.startswith("Q3(iii)")
)

df.loc[mask_q3_iii, "marks"] = 0.5

print(
    "✓ Q3(iii):",
    mask_q3_iii.sum(),
    "records × 0.5 mark"
)

# ============================================================
# 8. CONVERT MARKS TO NUMERIC
# ============================================================

df["marks"] = pd.to_numeric(
    df["marks"],
    errors="coerce"
)

# ============================================================
# 9. CHECK CURRENT Q1/Q2 QUESTION TEXT
# ============================================================

print("\n")
print("=" * 80)
print("CURRENT Q1/Q2 RECORDS")
print("=" * 80)

q1_q2 = df[
    df["question_id"]
    .astype(str)
    .str.match(r"ENG25-P1-Q[12]-", na=False)
]

display(
    q1_q2[
        [
            "question_id",
            "question_number",
            "question_type",
            "marks",
            "question_text",
            "answer"
        ]
    ]
)

# ============================================================
# 10. IMPORTANT SAFETY CHECK
# ============================================================

print("\n")
print("=" * 80)
print("QUESTION TEXT / ANSWER DUPLICATION CHECK")
print("=" * 80)

q1_q2_check = q1_q2.copy()

q1_q2_check["same_text"] = (
    q1_q2_check["question_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    ==
    q1_q2_check["answer"]
    .fillna("")
    .astype(str)
    .str.strip()
)

same_count = q1_q2_check["same_text"].sum()

print(
    "Q1/Q2 records where question_text == answer:",
    same_count
)

if same_count > 0:
    print("\n⚠ These records still need question-text reconstruction.")
    print(
        "The original PDF text was printed above so we can verify "
        "the exact question wording before replacing anything."
    )
else:
    print("✓ Q1/Q2 question_text and answer are different.")

# ============================================================
# 11. SAVE INTERMEDIATE CORRECTED FILE
# ============================================================

corrected_file = (
    "ISC_Class12_English_Paper1_2025_MASTER_CORRECTED.csv"
)

df.to_csv(
    corrected_file,
    index=False,
    encoding="utf-8"
)

print("\n")
print("=" * 80)
print("CORRECTED FILE SAVED")
print("=" * 80)

print(corrected_file)

# ============================================================
# 12. MARKS SUMMARY
# ============================================================

print("\n")
print("=" * 80)
print("UPDATED MARKS")
print("=" * 80)

print(
    df.groupby(
        df["question_number"]
        .fillna("")
        .astype(str)
        .str.extract(r"(Q[1-4])", expand=False)
    )["marks"]
    .sum()
)

print("\nRecord-level marks total:", df["marks"].sum())

print("\nPaper-level structure:")
print("Q1 = 20")
print("Q2 = 25")
print("Q3 = 15")
print("Q4 = 20")
print("Paper Maximum = 80")

In [ ]:
# ============================================================
# FIX Q1 & Q2 QUESTION TEXT
# ISC CLASS 12 ENGLISH PAPER 1 — 2025
# ============================================================

import pandas as pd
import os

# ============================================================
# 1. LOAD CORRECTED MASTER FILE
# ============================================================

input_file = "ISC_Class12_English_Paper1_2025_MASTER_CORRECTED.csv"

if not os.path.exists(input_file):
    raise FileNotFoundError(
        f"{input_file} was not found."
    )

df = pd.read_csv(input_file)

print("=" * 80)
print("LOADED CORRECTED MASTER DATASET")
print("=" * 80)

print("Records:", len(df))

# ============================================================
# 2. MAKE MARKS FLOAT
# ============================================================

df["marks"] = pd.to_numeric(
    df["marks"],
    errors="coerce"
).astype(float)

# ============================================================
# 3. CORRECT Q1 QUESTION TEXT
# ============================================================

q1_questions = {
    "ENG25-P1-Q1-i":
        "Write a composition (in approximately 400 - 450 words) on the following subject: "
        "The children’s library in your locality was in a neglected condition. You and your "
        "friends visited it recently and decided to do something about it. What did you see "
        "there and how did you bring about a significant change? Describe the excitement on "
        "the reopening of the library and how you felt when many children began to visit it eventually.",

    "ENG25-P1-Q1-ii":
        "Write a composition (in approximately 400 - 450 words) on the following subject: "
        "You depended on technology to complete an important task. However, it failed you "
        "at a crucial moment. Narrate what happened, your attempts to fix it and the lessons "
        "you learnt from this experience.",

    "ENG25-P1-Q1-iii":
        "Write a composition (in approximately 400 - 450 words) on the following subject: "
        "Film stars are good role models for youngsters. Argue either FOR or AGAINST the given statement.",

    "ENG25-P1-Q1-iv":
        "Write a composition (in approximately 400 - 450 words) on the following subject: "
        "Friendship",

    "ENG25-P1-Q1-v":
        "Write a composition (in approximately 400 - 450 words) on the following subject: "
        "Clothes define a person. Present your reflections on this statement.",

    "ENG25-P1-Q1-vi":
        "Write a composition (in approximately 400 - 450 words) on the following subject: "
        "Write an original short story which ends with the following words: "
        "He stood looking out of the window, with a smile on his face."
}

# ============================================================
# 4. CORRECT Q2 QUESTION TEXT
# ============================================================

q2_questions = {
    "ENG25-P1-Q2-i":
        "Write a feature article in about 300 words, to be published in your school magazine "
        "on the topic “The Importance of Sports”. Your article must use the guidelines given below: "
        "Inculcates discipline – builds teamwork – has health benefits – develops leadership qualities "
        "– leads to character building – provides career options.",

    "ENG25-P1-Q2-ii":
        "You are the President of The Nature Club of your school. You want to organise "
        "‘Tree Plantation Day’ in order to celebrate its tenth anniversary. Write a proposal "
        "in about 150 words stating the steps you would take to organise the programme and make it a success."
}

# ============================================================
# 5. UPDATE QUESTION TEXT
# ============================================================

print("\n")
print("=" * 80)
print("UPDATING QUESTION TEXT")
print("=" * 80)

updated_count = 0

for question_id, question_text in {**q1_questions, **q2_questions}.items():

    mask = df["question_id"].astype(str) == question_id

    if mask.any():
        df.loc[mask, "question_text"] = question_text
        updated_count += mask.sum()
        print(f"✓ Updated: {question_id}")
    else:
        print(f"⚠ Question ID not found: {question_id}")

print("\nTotal Q1/Q2 records updated:", updated_count)

# ============================================================
# 6. CORRECT Q2 MARKS
# ============================================================

df.loc[
    df["question_id"] == "ENG25-P1-Q2-i",
    "marks"
] = 15.0

df.loc[
    df["question_id"] == "ENG25-P1-Q2-ii",
    "marks"
] = 10.0

print("\n✓ Q2 marks confirmed: 15 + 10 = 25")

# ============================================================
# 7. CORRECT Q3 MARK ALLOCATION
# ============================================================

# Q3(i): 5 questions × 1 mark
mask_q3_i = (
    df["question_number"]
    .fillna("")
    .astype(str)
    .str.startswith("Q3(i)")
)

df.loc[mask_q3_i, "marks"] = 1.0

# Q3(ii): 10 questions × 0.5 mark
mask_q3_ii = (
    df["question_number"]
    .fillna("")
    .astype(str)
    .str.startswith("Q3(ii)")
)

df.loc[mask_q3_ii, "marks"] = 0.5

# Q3(iii): 10 questions × 0.5 mark
mask_q3_iii = (
    df["question_number"]
    .fillna("")
    .astype(str)
    .str.startswith("Q3(iii)")
)

df.loc[mask_q3_iii, "marks"] = 0.5

print("✓ Q3(i) marks corrected")
print("✓ Q3(ii) marks corrected")
print("✓ Q3(iii) marks corrected")

# ============================================================
# 8. CHECK Q1/Q2 QUESTION VS ANSWER
# ============================================================

q1_q2 = df[
    df["question_id"]
    .astype(str)
    .str.match(r"ENG25-P1-Q[12]-", na=False)
].copy()

q1_q2["same_text"] = (
    q1_q2["question_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    ==
    q1_q2["answer"]
    .fillna("")
    .astype(str)
    .str.strip()
)

same_count = q1_q2["same_text"].sum()

print("\n")
print("=" * 80)
print("Q1/Q2 QUESTION-ANSWER CHECK")
print("=" * 80)

print("Q1/Q2 records:", len(q1_q2))
print("Question text = answer:", same_count)

if same_count == 0:
    print("✓ Q1/Q2 question text and answers are now separated.")
else:
    print("⚠ Some Q1/Q2 records still have identical question text and answers.")
    display(
        q1_q2[q1_q2["same_text"]][
            [
                "question_id",
                "question_text",
                "answer"
            ]
        ]
    )

# ============================================================
# 9. CHECK EXACT DUPLICATE QUESTION TEXT
# ============================================================

duplicate_questions = df[
    df["question_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .duplicated(keep=False)
]

print("\n")
print("=" * 80)
print("DUPLICATE QUESTION TEXT CHECK")
print("=" * 80)

print(
    "Potential exact duplicate records:",
    len(duplicate_questions)
)

if len(duplicate_questions) == 0:
    print("✓ No exact duplicate question texts found.")
else:
    print("⚠ Potential duplicates found:")
    display(
        duplicate_questions[
            [
                "question_id",
                "question_number",
                "question_type",
                "question_text"
            ]
        ]
    )

# ============================================================
# 10. NULL CHECK
# ============================================================

print("\n")
print("=" * 80)
print("NULL CHECK")
print("=" * 80)

missing = df.isnull().sum()

if missing.sum() == 0:
    print("✓ No NULL values.")
else:
    print("Columns containing NULL values:")
    print(missing[missing > 0])

# ============================================================
# 11. DUPLICATE ID CHECK
# ============================================================

print("\n")
print("=" * 80)
print("QUESTION ID CHECK")
print("=" * 80)

duplicate_ids = df[
    df["question_id"].duplicated(keep=False)
]

print("Duplicate question ID records:", len(duplicate_ids))

if len(duplicate_ids) == 0:
    print("✓ No duplicate question IDs.")
else:
    display(
        duplicate_ids[
            ["question_id", "question_number"]
        ]
    )

# ============================================================
# 12. RECORD COUNT
# ============================================================

print("\n")
print("=" * 80)
print("RECORD COUNT")
print("=" * 80)

print("Total records:", len(df))

if len(df) == 43:
    print("✓ Expected 43 records.")
else:
    print("⚠ Expected 43 records.")

# ============================================================
# 13. QUESTION TYPE DISTRIBUTION
# ============================================================

print("\n")
print("=" * 80)
print("QUESTION TYPE DISTRIBUTION")
print("=" * 80)

print(
    df["question_type"]
    .value_counts()
)

# ============================================================
# 14. FINAL PAPER STRUCTURE
# ============================================================

print("\n")
print("=" * 80)
print("PAPER-LEVEL MARK STRUCTURE")
print("=" * 80)

print("Q1 = 20 marks")
print("Q2 = 25 marks")
print("Q3 = 15 marks")
print("Q4 = 20 marks")
print("-" * 40)
print("TOTAL = 80 marks")

# ============================================================
# 15. SAVE FINAL CLEAN DATASET
# ============================================================

final_file = (
    "ISC_Class12_English_Paper1_2025_FINAL_CLEAN.csv"
)

df.to_csv(
    final_file,
    index=False,
    encoding="utf-8"
)

print("\n")
print("=" * 80)
print("FINAL FILE SAVED")
print("=" * 80)

print("✓", final_file)
print("✓ Records:", len(df))
print("✓ Columns:", len(df.columns))

# ============================================================
# 16. DISPLAY Q1/Q2 AFTER CORRECTION
# ============================================================

print("\n")
print("=" * 80)
print("FINAL Q1/Q2 RECORDS")
print("=" * 80)

display(
    df[
        df["question_id"]
        .astype(str)
        .str.match(r"ENG25-P1-Q[12]-", na=False)
    ][
        [
            "question_id",
            "question_type",
            "marks",
            "question_text",
            "answer"
        ]
    ]
)

In [ ]:
import pandas as pd
import os

# ============================================================
# 1. LOAD FINAL CLEAN FILE
# ============================================================

input_file = "ISC_Class12_English_Paper1_2025_FINAL_CLEAN.csv"

if not os.path.exists(input_file):
    raise FileNotFoundError(f"{input_file} was not found.")

df = pd.read_csv(input_file)

print("=" * 80)
print("STANDARDIZING QUESTION METADATA")
print("=" * 80)

print("Records before standardization:", len(df))

# ============================================================
# 2. CREATE STANDARD PARENT QUESTION
# ============================================================

# Extract Q1, Q2, Q3 or Q4 from question_number/question_id

def get_parent_question(row):
    qid = str(row["question_id"])

    match = pd.Series(qid).str.extract(
        r"(Q[1-4])",
        expand=False
    ).iloc[0]

    return match if pd.notna(match) else None


df["parent_question"] = df.apply(
    get_parent_question,
    axis=1
)

# ============================================================
# 3. CREATE STANDARD QUESTION NUMBER
# ============================================================

# Use question_id because it is our clean, unique identifier.
# Convert:
#
# ENG25-P1-Q1-i     -> Q1(i)
# ENG25-P1-Q1-ii    -> Q1(ii)
# ENG25-P1-Q3-i-a   -> Q3(i)(a)
# ENG25-P1-Q4-ii-a  -> Q4(ii)(a)

def create_question_number(question_id):

    qid = str(question_id)

    # Q1 / Q2
    match = pd.Series(qid).str.extract(
        r"Q([1-4])-([ivx]+)$",
        expand=True
    )

    if pd.notna(match.iloc[0, 0]) and pd.notna(match.iloc[0, 1]):
        return f"Q{match.iloc[0,0]}({match.iloc[0,1]})"

    # Q3 / Q4 with two levels
    match = pd.Series(qid).str.extract(
        r"Q([1-4])-([ivx]+)-([a-j0-9]+)$",
        expand=True
    )

    if (
        pd.notna(match.iloc[0, 0])
        and pd.notna(match.iloc[0, 1])
        and pd.notna(match.iloc[0, 2])
    ):
        return (
            f"Q{match.iloc[0,0]}"
            f"({match.iloc[0,1]})"
            f"({match.iloc[0,2]})"
        )

    return None


df["question_number"] = df["question_id"].apply(
    create_question_number
)

# ============================================================
# 4. CREATE STANDARD SUB-QUESTION
# ============================================================

def create_sub_question(question_id):

    qid = str(question_id)

    # Q1-i / Q2-i
    match = pd.Series(qid).str.extract(
        r"Q[1-4]-([ivx]+)$",
        expand=False
    ).iloc[0]

    if pd.notna(match):
        return str(match)

    # Q3-i-a / Q4-ii-a
    match = pd.Series(qid).str.extract(
        r"Q[1-4]-([ivx]+)-([a-j0-9]+)$",
        expand=True
    )

    if pd.notna(match.iloc[0, 0]) and pd.notna(match.iloc[0, 1]):
        return f"{match.iloc[0,0]}-{match.iloc[0,1]}"

    return None


df["sub_question"] = df["question_id"].apply(
    create_sub_question
)

# ============================================================
# 5. STANDARDIZE INSTRUCTIONS
# ============================================================

# Keep the actual question-specific instruction where available.
# For records where instruction is missing, create a meaningful
# section-level instruction instead of leaving it blank.

def assign_instruction(row):

    qtype = str(row["question_type"])
    qnum = str(row["question_number"])

    # Q1
    if qnum.startswith("Q1"):
        return (
            "Write a composition in approximately 400-450 words "
            "on any one of the given subjects."
        )

    # Q2 Feature Article
    if qnum == "Q2(i)":
        return (
            "Write a feature article in about 300 words for the "
            "school magazine using the given guidelines."
        )

    # Q2 Proposal
    if qnum == "Q2(ii)":
        return (
            "Write a proposal in about 150 words stating the steps "
            "you would take to organise the programme and make it a success."
        )

    # Q3(i)
    if qnum.startswith("Q3(i)"):
        return (
            "Rewrite the sentence according to the instruction "
            "given after it."
        )

    # Q3(ii)
    if qnum.startswith("Q3(ii)"):
        return (
            "Replace the incorrect underlined word to make the "
            "sentence meaningful and grammatically correct. "
            "Do not copy the sentence."
        )

    # Q3(iii)
    if qnum.startswith("Q3(iii)"):
        return (
            "Fill in the blank with the appropriate form of the "
            "verb given in brackets. Write the verbs in the correct order."
        )

    # Q4(i)(a)
    if qnum.startswith("Q4(i)(a)"):
        return (
            "Give a word or phrase that can replace the given "
            "word without changing the meaning."
        )

    # Q4(i)(b)
    if qnum.startswith("Q4(i)(b)"):
        return (
            "Choose the option in which the word is used with "
            "a different meaning."
        )

    # Q4(ii)
    if qnum.startswith("Q4(ii)"):
        return (
            "Answer the following questions based on the passage."
        )

    # Q4(iii)
    if qnum.startswith("Q4(iii)"):
        return (
            "Summarise the passage in approximately 100 words."
        )

    # Preserve existing instruction if none of the above applies
    existing = row.get("instruction", "")

    if pd.notna(existing) and str(existing).strip():
        return str(existing).strip()

    return ""


df["instruction"] = df.apply(
    assign_instruction,
    axis=1
)

# ============================================================
# 6. STANDARDIZE CLASS / SUBJECT / PAPER
# ============================================================

df["class"] = df["class"].replace({
    "Class XII": "12"
})

df["subject"] = df["subject"].replace({
    "English": "English Language"
})

df["paper"] = df["paper"].replace({
    "Paper 1 - English Language": "Paper 1"
})

# ============================================================
# 7. STANDARDIZE COLUMN ORDER
# ============================================================

standard_columns = [
    "question_id",
    "board",
    "class",
    "subject",
    "year",
    "paper",
    "source_type",
    "parent_question",
    "sub_question",
    "question_type",
    "marks",
    "question_text",
    "answer",
    "source_file",
    "question_number",
    "instruction"
]

df = df[standard_columns]

# ============================================================
# 8. VALIDATION
# ============================================================

print("\n")
print("=" * 80)
print("METADATA VALIDATION")
print("=" * 80)

print("\nParent question:")
print(df["parent_question"].value_counts().sort_index())

print("\nMissing parent_question:")
print(df["parent_question"].isna().sum())

print("\nMissing sub_question:")
print(df["sub_question"].isna().sum())

print("\nMissing question_number:")
print(df["question_number"].isna().sum())

print("\nMissing instruction:")
print(
    df["instruction"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

# ============================================================
# 9. DISPLAY METADATA SAMPLE
# ============================================================

print("\n")
print("=" * 80)
print("STANDARDIZED METADATA SAMPLE")
print("=" * 80)

display(
    df[
        [
            "question_id",
            "parent_question",
            "sub_question",
            "question_number",
            "question_type",
            "marks",
            "instruction"
        ]
    ].head(15)
)

# ============================================================
# 10. CHECK QUESTION IDs
# ============================================================

duplicate_ids = df[
    df["question_id"].duplicated(keep=False)
]

print("\n")
print("=" * 80)
print("QUESTION ID CHECK")
print("=" * 80)

print("Total IDs:", df["question_id"].nunique())
print("Duplicate ID records:", len(duplicate_ids))

if len(duplicate_ids) == 0:
    print("✓ No duplicate question IDs.")
else:
    display(duplicate_ids)

# ============================================================
# 11. CHECK QUESTION TEXT
# ============================================================

empty_questions = df[
    df["question_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
]

print("\n")
print("=" * 80)
print("QUESTION TEXT CHECK")
print("=" * 80)

print("Empty question records:", len(empty_questions))

if len(empty_questions) == 0:
    print("✓ All questions have question text.")
else:
    display(empty_questions)

# ============================================================
# 12. CHECK ANSWERS
# ============================================================

empty_answers = df[
    df["answer"]
    .fillna("")
    .astype(str)
    .str.strip()
    .eq("")
]

print("\n")
print("=" * 80)
print("ANSWER CHECK")
print("=" * 80)

print("Empty answer records:", len(empty_answers))

if len(empty_answers) == 0:
    print("✓ All records have answers.")
else:
    display(empty_answers)

# ============================================================
# 13. PAPER STRUCTURE
# ============================================================

print("\n")
print("=" * 80)
print("FINAL PAPER STRUCTURE")
print("=" * 80)

print("Q1 = 20 marks")
print("Q2 = 25 marks")
print("Q3 = 15 marks")
print("Q4 = 20 marks")
print("-" * 40)
print("TOTAL = 80 marks")

# ============================================================
# 14. SAVE STANDARDIZED DATASET
# ============================================================

final_file = (
    "ISC_Class12_English_Paper1_2025_DATABASE_READY.csv"
)

df.to_csv(
    final_file,
    index=False,
    encoding="utf-8"
)

print("\n")
print("=" * 80)
print("DATABASE-READY FILE")
print("=" * 80)

print("✓ File saved:", final_file)
print("✓ Records:", len(df))
print("✓ Columns:", len(df.columns))

In [ ]:
import pandas as pd
import os

# Original database-ready CSV
input_file = "ISC_Class12_English_Paper1_2025_DATABASE_READY.csv"

# New UTF-8-SIG version for MySQL Workbench
output_file = "ISC_Class12_English_Paper1_2025_MYSQL_IMPORT.csv"

# Load the CSV
df = pd.read_csv(input_file)

# Make sure marks can contain 0.5
df["marks"] = pd.to_numeric(df["marks"], errors="coerce")

# Save using UTF-8 with BOM
df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("MYSQL IMPORT FILE CREATED")
print("=" * 70)

print("File:", output_file)
print("Records:", len(df))
print("Columns:", len(df.columns))

# Verify that the file can be read back
test_df = pd.read_csv(
    output_file,
    encoding="utf-8-sig"
)

print("\nVerification:")
print("Records read back:", len(test_df))
print("Columns read back:", len(test_df.columns))

if len(test_df) == 43 and len(test_df.columns) == 16:
    print("✓ File verified successfully.")
else:
    print("⚠ File verification failed.")

print("\n✓ Ready for MySQL Workbench import.")

In [ ]:
import pandas as pd
import os

# ============================================================
# CREATE MYSQL SQL IMPORT FILE
# ============================================================

csv_file = "ISC_Class12_English_Paper1_2025_DATABASE_READY.csv"
sql_file = "ISC_Class12_English_Paper1_2025_MYSQL_IMPORT.sql"

if not os.path.exists(csv_file):
    raise FileNotFoundError(f"{csv_file} not found.")

df = pd.read_csv(csv_file)

# Make sure marks supports 0.5
df["marks"] = pd.to_numeric(df["marks"], errors="coerce")


# ------------------------------------------------------------
# MySQL string escaping
# ------------------------------------------------------------

def mysql_escape(value):

    if pd.isna(value):
        return "NULL"

    value = str(value)

    # Escape characters required by MySQL
    value = value.replace("\\", "\\\\")
    value = value.replace("'", "\\'")
    value = value.replace("\x00", "\\0")
    value = value.replace("\n", "\\n")
    value = value.replace("\r", "\\r")

    return "'" + value + "'"


# ------------------------------------------------------------
# Columns in exact database order
# ------------------------------------------------------------

columns = [
    "question_id",
    "board",
    "class",
    "subject",
    "year",
    "paper",
    "source_type",
    "parent_question",
    "sub_question",
    "question_type",
    "marks",
    "question_text",
    "answer",
    "source_file",
    "question_number",
    "instruction"
]


# ------------------------------------------------------------
# Start SQL file
# ------------------------------------------------------------

sql_lines = []

sql_lines.append("USE isc_assessment;")
sql_lines.append("")
sql_lines.append("SET NAMES utf8mb4;")
sql_lines.append("")
sql_lines.append("SET FOREIGN_KEY_CHECKS = 0;")
sql_lines.append("")
sql_lines.append("START TRANSACTION;")
sql_lines.append("")


# ------------------------------------------------------------
# INSERT records
# ------------------------------------------------------------

for _, row in df.iterrows():

    values = []

    for col in columns:

        value = row[col]

        # Numeric fields
        if col == "year":
            if pd.isna(value):
                values.append("NULL")
            else:
                values.append(str(int(value)))

        elif col == "marks":
            if pd.isna(value):
                values.append("NULL")
            else:
                values.append(str(float(value)))

        # Text fields
        else:
            values.append(mysql_escape(value))

    sql = (
        "INSERT INTO questions ("
        + ", ".join(f"`{col}`" for col in columns)
        + ") VALUES ("
        + ", ".join(values)
        + ");"
    )

    sql_lines.append(sql)


# ------------------------------------------------------------
# Finish SQL file
# ------------------------------------------------------------

sql_lines.append("")
sql_lines.append("COMMIT;")
sql_lines.append("")
sql_lines.append("SET FOREIGN_KEY_CHECKS = 1;")
sql_lines.append("")

# Write UTF-8 SQL file
with open(sql_file, "w", encoding="utf-8") as f:
    f.write("\n".join(sql_lines))


# ============================================================
# VALIDATION
# ============================================================

print("=" * 80)
print("MYSQL SQL IMPORT FILE CREATED")
print("=" * 80)

print("CSV records:", len(df))
print("CSV columns:", len(df.columns))
print("SQL INSERT statements:", len(df))

print("\nSQL file:")
print(sql_file)

if len(df) == 43:
    print("\n✓ 43 records converted successfully.")
else:
    print(f"\n⚠ Expected 43 records, found {len(df)}.")

print("\n✓ UTF-8 encoding used.")
print("✓ Unicode characters preserved.")
print("✓ Ready to execute in MySQL Workbench.")